# CoT
LLM이 답변하기 위해서는 연관성 있는 문서만을 참고해서 답변을 수행해야 합니다. 이 때 답변하기 전에 **근거**부터 작성하라고 하는 프롬프트 엔지니어링 기법을 CoT(Chain of Thought)라고 합니다. 파인튜닝 수행 시에도 근거부터 작성하고 답변하는 식으로 파인튜닝 합니다. 마치 수학 문제를 풀기 위해 중간 중간 풀이 과정을 쓰는 것 처럼 생각할 수 있습니다.

CoT는 크게 Orion 방식과 Cohere 방식이 있습니다. 두 방식 모두 원래는 모델의 이름입니다.
- Orion 방식은 인용한 문서의 번호를 답변 뒤에 붙여주는 방식입니다.
  ```
  ["Source_id": 0, "Content": "황제 펭귄은 세계에서 가장 키가 큰 펭귄입니다. 또한, 삶의 대부분을 남극해에서 보내며, 모든 새 중 가장 깊은 수심인 500~600m 까지 잠수할 수 있다. 주로 작은 어류와 오징어를 사냥합니다."]
["Source_id": 1, "Content": "황제 펭귄은 남극에서 주로 서식하며 성체의 키가 120cm, 수명은 약 20년, 체중이 23kg에서 최대 45kg에까지 달합니다."]
["Source_id": 2, "Content": "꼬마펭귄 핑구에 등장하는 핑구의 여동생 핑가가 황제펭귄 새끼와 비슷하게 생겼다. 그 외에도 단역으로 나오는 아기 펭귄들도 황제펭귄 새끼의 모습이다. 그런데 핑구를 비롯한 나머지 펭귄들은 황제펭귄 특유의 귀의 노란 무늬가 없는 것으로 보아 아델리 펭귄을 모티브로 한 것으로 보인다. 참고로 아델리 펭귄과 황제펭귄은 서로 숙적이자 견원지간 사이므로 서로 만나려고 하지 않는다. 그나마 아빠는 황제펭귄과 외형이 닮았다. 그런데 잘 보면 핑구도 그렇고 가족 전부 흰 배가 노란끼가 살짝 도는 걸 보면 황제펭귄이 맞는 걸로 보인다."]
질문: 남극에서 가장 큰 펭귄은?
답변: 남극에서 가장 큰 펭귄은 황제 펭귄이며 세계에서 가장 키가 큰 펭귄입니다. 남극에서 주로 서식하며 성체의 키가 120cm 체중이 23kg에서 최대 45kg에까지 달합니다.[[0], [1]]```

- Cohere 방식은 답변 중간에 어떤 문서에서 인용했는지를 태그 같은 것을 이용해서 작성하는 방식입니다. Orion 방식보다 조금 더 효과가 좋다고 알려져 있으나, 프롬프트를 작성하는 것이 쉽지는 않습니다.
```
["Source_id": 0, "Content": "황제 펭귄은 세계에서 가장 키가 큰 펭귄입니다. 또한, 삶의 대부분을 남극해에서 보내며, 모든 새 중 가장 깊은 수심인 500~600m 까지 잠수할 수 있다. 주로 작은 어류와 오징어를 사냥합니다."]
["Source_id": 1, "Content": "황제 펭귄은 남극에서 주로 서식하며 성체의 키가 120cm, 수명은 약 20년, 체중이 23kg에서 최대 45kg에까지 달합니다."]
["Source_id": 2, "Content": "꼬마펭귄 핑구에 등장하는 핑구의 여동생 핑가가 황제펭귄 새끼와 비슷하게 생겼다. 그 외에도 단역으로 나오는 아기 펭귄들도 황제펭귄 새끼의 모습이다. 그런데 핑구를 비롯한 나머지 펭귄들은 황제펭귄 특유의 귀의 노란 무늬가 없는 것으로 보아 아델리 펭귄을 모티브로 한 것으로 보인다. 참고로 아델리 펭귄과 황제펭귄은 서로 숙적이자 견원지간 사이므로 서로 만나려고 하지 않는다. 그나마 아빠는 황제펭귄과 외형이 닮았다. 그런데 잘 보면 핑구도 그렇고 가족 전부 흰 배가 노란끼가 살짝 도는 걸 보면 황제펭귄이 맞는 걸로 보인다."]
질문: 남극에서 가장 큰 펭귄은?
답변: 남극에서 가장 큰 펭귄은 <co: 0>황제 펭귄이며 세계에서 가장 키가 큰 펭귄</co: 0>입니다. <co: 1>남극에서 주로 서식</co: 1>하며 <co: 1>성체의 키가 120cm 체중이 23kg에서 최대 45kg</co: 1>에까지 달합니다.
```

  

# RAFT
RAFT는 UC Berkely 대학교의 논문으로서 사용자의 질문과 연관된 문서와 정답과 연관이 없는 문서(negative documents)를 섞은 데이터 세트를 구축하는 방법입니다. 연관된 문서와 연관되어있지 않은 문서들이 모두 주어졌을 때 답변하는 연습을 하는 식으로 파인튜닝 되며, LLM의 능력을 크게 향상시킵니다.

In [1]:
!nvidia-smi

Tue Oct 15 04:05:28 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.104.05             Driver Version: 535.104.05   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   37C    P8               8W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

# 데이터 로딩

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
SQUAD_ANSWER_FILE_PATH="/content/drive/MyDrive/HANKYUNG_WITH_TOSS_BANK-2(소민호 강사님)/week17/data/ko_nia_normal_squad_all.json"
SQUAD_NO_ANSWER_FILE_PATH="/content/drive/MyDrive/HANKYUNG_WITH_TOSS_BANK-2(소민호 강사님)/week17/data/ko_nia_noanswer_squad_all.json"

In [7]:
import os
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path

from sklearn.model_selection import train_test_split

def read_squad(path):

    path = Path(path)

    with open(path, 'rb') as f:
        squad_dict = json.load(f)

    contexts = []
    questions = []
    answers = []
    group_numbers = []

    for group in squad_dict['data']:
        # if group['source'] != 2:
            # continue
        group_number = group['source']

        for passage in group['paragraphs']:
            context = passage['context']

            for qa in passage['qas']:
                question = qa['question']

                for answer in qa['answers']:
                    contexts.append(context)
                    questions.append(question)
                    answers.append(answer)
                    group_numbers.append(group_number)

    return contexts, questions, answers, group_numbers

total_contexts, total_questions, total_answers, group_numbers = read_squad(SQUAD_ANSWER_FILE_PATH)

In [ ]:
# group: squad_dict['data']는 여러 질문과 답변의 그룹을 담고 있는 리스트입니다. 각 그룹에는 여러 본문(passage)과 질문-답변 쌍이 포함됩니다.
# 본문(Context) 추출: 각 passage에서 context 값을 추출하여 contexts 리스트에 추가합니다.
# 질문(Question) 추출: 각 본문 안에는 여러 qas(질문-답변 쌍)가 있습니다. 각 질문을 questions 리스트에 추가합니다.
# 답변(Answer) 추출: 각 질문에 대한 여러 개의 답변 중, 답변을 answers 리스트에 추가합니다.
# 그룹 번호 저장: 각 질문-답변 쌍이 속한 그룹 번호(카테고리 등)를 group_numbers 리스트에 저장합니다.

# 함수의 전체 흐름

# SQuAD JSON 파일을 열어 데이터를 읽고 파싱합니다.
# 본문, 질문, 답변을 각각의 리스트에 저장하고, 질문이 속한 그룹 정보도 함께 저장합니다.
# 본문, 질문, 답변 데이터를 각각 별도로 리스트로 반환합니다.

# 구조
# {
#   "data": [
#     {
#       "source": 1,
#       "paragraphs": [
#         {
#           "context": "본문 내용",
#           "qas": [
#             {
#               "question": "질문 내용",
#               "answers": [
#                 {
#                   "text": "답변 내용",
#                   "answer_start": 123
#                 }
#               ]
#             }
#           ]
#         }
#       ]
#     }
#   ]
# }

```
source
1	정치
2	경제
3	사회
4	생활
5	IT/과학
6	연예
7	스포츠
8	문화
9	미용/건강
```

In [8]:
total_contexts[0]

"한국청소년단체협의회와 여성가족부는 22일부터 28일까지 서울과 충북 괴산에서 '국제청소년포럼'을 연다고 21일 밝혔다. 한국 미국 캐나다 호주 등 전 세계 32개국 75여명의 대학생, 청소년들이 모여 전 세계적 현안문제에 대한 대안과 해결책을 모색하는 자리다. 이번 포럼의 주제는 '청소년과 뉴미디어'다. 스마트폰 SNS 태블릿PC 등 새로운 커뮤니케이션 매체인 '뉴미디어'에 대한 성찰과 문제점에 대해 토론한다. 기조강연을 시작으로 국가별 주제관련 사례발표, 그룹 토론 및 전체총회, '청소년선언문' 작성 및 채택 등 다양한 프로그램을 운영한다. 개회식은 22일 서울 방화동에 있는 국제청소년센터 국제회의장에서 한다. 전 세계 32개국 대학생ㆍ청소년 참가자와 전국의 청소년기관단체장과 청소년지도자 여성가족부 주한외교사절 등 100여명이 참석할 예정이다. 23일에는 유엔미래포럼 박영숙 대표가 '뉴미디어의 균형 있는 발전을 위한 청소년의 역할'에 대해 기조강연을 한다. 뉴미디어의 올바른 활용방안과 청소년문화의 형성에 대해 설명할 계획이다. 27일 폐회식에서는 '청소년선언문'을 채택한다. 선언문에는 전 세계적으로 뉴미디어의 바람직한 발전을 촉구하며 각국 청년들이 함께 실천할 수 있는 내용 등이 담길 예정이다. 한국청소년단체협의회는 포럼이 끝난 뒤 UN 등 국제기구와 참가자 각국 정부 등 국제사회에 선언문을 전달할 예정이다."

In [9]:
total_questions[0]

"서울과 충북 괴산에서 '국제청소년포럼'을 여는 곳은?"

In [10]:
total_answers[0]

{'answer_start': 0, 'text': '한국청소년단체협의회와 여성가족부'}

In [11]:
def add_end_idx(answers, contexts):
    for answer, context in zip(answers, contexts):
        answer['text'] = answer['text'].rstrip()
        gold_text = answer['text']
        start_idx = answer['answer_start']
        end_idx = start_idx + len(gold_text)

        assert context[start_idx:end_idx] == gold_text, "end_index 계산에 에러가 있습니다."
        answer['answer_end'] = end_idx

add_end_idx(total_answers, total_contexts)

In [12]:
df = pd.DataFrame({'contexts': total_contexts, 'questions': total_questions, 'answers': total_answers, 'group_id': group_numbers})
df.head()

,contexts,questions,answers,group_id
0,한국청소년단체협의회와 여성가족부는 22일부터 28일까지 서울과 충북 괴산에서 '국제...,서울과 충북 괴산에서 '국제청소년포럼'을 여는 곳은?,"{'answer_start': 0, 'text': '한국청소년단체협의회와 여성가족부...",5
1,한국청소년단체협의회와 여성가족부는 22일부터 28일까지 서울과 충북 괴산에서 '국제...,'국제 청소년포럼'이 열리는 때는?,"{'answer_start': 19, 'text': '22일부터 28일', 'ans...",5
2,한국청소년단체협의회와 여성가족부는 22일부터 28일까지 서울과 충북 괴산에서 '국제...,이번 포럼의 주제는?,"{'answer_start': 157, 'text': ''청소년과 뉴미디어'', '...",5
3,한국청소년단체협의회와 여성가족부는 22일부터 28일까지 서울과 충북 괴산에서 '국제...,포럼은 어떻게 진행되는가?,"{'answer_start': 232, 'text': '기조강연을 시작으로 국가별 ...",5
4,[헤럴드POP=고승아 기자]그룹 구구단 샐리가 리듬체조에서 실수했다.15일 방송된 ...,아육대에서 리듬체조에 출전한 구구단의 멤버는?,"{'answer_start': 22, 'text': '샐리', 'answer_end...",7


In [13]:
# 본문 중복 제거
df = df.drop_duplicates(subset='contexts')
df.head()

,contexts,questions,answers,group_id
0,한국청소년단체협의회와 여성가족부는 22일부터 28일까지 서울과 충북 괴산에서 '국제...,서울과 충북 괴산에서 '국제청소년포럼'을 여는 곳은?,"{'answer_start': 0, 'text': '한국청소년단체협의회와 여성가족부...",5
4,[헤럴드POP=고승아 기자]그룹 구구단 샐리가 리듬체조에서 실수했다.15일 방송된 ...,아육대에서 리듬체조에 출전한 구구단의 멤버는?,"{'answer_start': 22, 'text': '샐리', 'answer_end...",7
5,중국의 한 여성 경찰이 아파트에서 추락하던 3세 아이를 살리고 자신은 혼수상태에 빠...,중국에서 아파트에서 추락하던 3세 아이를 살리고 자신은 혼수상태에 빠진 사람은 누구야?,"{'answer_start': 134, 'text': '보조 교통 경찰로 일하는 천...",4
12,"[[the300]기업들 행정가처분 소송도 증가, 조달청 패소율 86%]조달청이 부정...",부정당업자를 제재하는 '부정당제재'를 행하는 정부 기관은?,"{'answer_start': 39, 'text': '조달청', 'answer_en...",3
17,◈ NASA의 화성 탐사 로버인 오퍼튜니티(Opportunity)가 화성 표면에서 ...,화성 탐사 로버인 오퍼튜니티는 누가 만든거야?,"{'answer_start': 2, 'text': 'NASA', 'answer_en...",5


In [14]:
# 각 group_id에서 상위 500개만 남긴다.
result_df = pd.DataFrame()

for group_id in df['group_id'].unique():
    subset = df[df['group_id'] == group_id].head(500)
    result_df = pd.concat([result_df, subset])

result_df = result_df.reset_index(drop=True)

# 결과 데이터프레임 확인
result_df['group_id'].value_counts()

,count
group_id,
5,500
7,500
4,500
3,500
2,500
6,500
9,500
1,500
8,500


# ChatGPT를 이용한 답변 데이터 구축

In [15]:
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.2/325.2 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 5.6 MB/s eta 0:00:00


In [1]:
import openai

client = openai.OpenAI(api_key="~~~~~~~~~~~"
)

In [17]:
# 단답을 배제하고 긴 답변을 할 수 있도록 설정
system_prompt = """당신은 본문으로부터 질문의 답변을 작성하는 언어모델입니다.
본문에 질문의 답이 없는 경우 찾을 수 없다고 답변하세요. 너무 짧게 답변하지마세요. 답변은 세 줄 이상 작성하세요."""

In [18]:
# 데이터 프레임에 있는 본문(질문과 답변을 이용해 user prompt를 생성
user_prompt = []
for c, q in zip(result_df['contexts'].to_list(), result_df['questions'].to_list()):
  user_prompt.append('본문: ' + c + '\n' + '질문: ' + q + '\n답변:')

In [19]:
print(user_prompt[0])

본문: 한국청소년단체협의회와 여성가족부는 22일부터 28일까지 서울과 충북 괴산에서 '국제청소년포럼'을 연다고 21일 밝혔다. 한국 미국 캐나다 호주 등 전 세계 32개국 75여명의 대학생, 청소년들이 모여 전 세계적 현안문제에 대한 대안과 해결책을 모색하는 자리다. 이번 포럼의 주제는 '청소년과 뉴미디어'다. 스마트폰 SNS 태블릿PC 등 새로운 커뮤니케이션 매체인 '뉴미디어'에 대한 성찰과 문제점에 대해 토론한다. 기조강연을 시작으로 국가별 주제관련 사례발표, 그룹 토론 및 전체총회, '청소년선언문' 작성 및 채택 등 다양한 프로그램을 운영한다. 개회식은 22일 서울 방화동에 있는 국제청소년센터 국제회의장에서 한다. 전 세계 32개국 대학생ㆍ청소년 참가자와 전국의 청소년기관단체장과 청소년지도자 여성가족부 주한외교사절 등 100여명이 참석할 예정이다. 23일에는 유엔미래포럼 박영숙 대표가 '뉴미디어의 균형 있는 발전을 위한 청소년의 역할'에 대해 기조강연을 한다. 뉴미디어의 올바른 활용방안과 청소년문화의 형성에 대해 설명할 계획이다. 27일 폐회식에서는 '청소년선언문'을 채택한다. 선언문에는 전 세계적으로 뉴미디어의 바람직한 발전을 촉구하며 각국 청년들이 함께 실천할 수 있는 내용 등이 담길 예정이다. 한국청소년단체협의회는 포럼이 끝난 뒤 UN 등 국제기구와 참가자 각국 정부 등 국제사회에 선언문을 전달할 예정이다.
질문: 서울과 충북 괴산에서 '국제청소년포럼'을 여는 곳은?
답변:


In [20]:
# gpt에게 질문에 대한 답을 응답 받기
response = client.chat.completions.create(
    model = 'gpt-4o',
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt[0]}
    ]
)

In [21]:
response.choices[0].message.content

"'국제청소년포럼'을 여는 곳은 한국청소년단체협의회와 여성가족부입니다. 이 두 기관은 서울과 충북 괴산에서 행사를 개최하며 전 세계 32개국의 대학생과 청소년들이 참여하는 포럼을 진행합니다. 이번 포럼에서는 '청소년과 뉴미디어'라는 주제로 다양한 활동과 논의를 펼칠 예정입니다."

## 답변이 없는 데이터 세트 구축
negative sample documents를 위한 데이터 세트를 구축합니다. LLM 파인튜닝 및 RAG를 구축할 때 훨씬 더 효과가 좋습니다.

In [22]:
def read_squad_no_answer(path):
    path = Path(path)
    with open(path, 'rb') as f:
        squad_dict = json.load(f)

    contexts = []
    questions = []
    # answers = []
    group_numbers = []

    for group in squad_dict['data']:
        group_number = group['source']
        for passage in group['paragraphs']:
            context = passage['context']
            for qa in passage['qas']:
                question = qa['question']
                '''
                for answer in qa['answers']:
                    contexts.append(context)
                    questions.append(question)
                    answers.append(answer)
                    group_numbers.append(group_number)
                '''
                contexts.append(context)
                questions.append(question)
                group_numbers.append(group_number)

    return contexts, questions, group_numbers

In [23]:
total_contexts, total_questions, group_numbers = read_squad_no_answer(SQUAD_NO_ANSWER_FILE_PATH)

In [24]:
total_contexts[0]

'공정거래위원회는 상품권 발행비용 명목으로 고율의 발행수수료를 가맹점주에게 부담토록 한 제너시스비비큐(BBQ)에게 시정명령을 내렸다. 공정위에 따르면 제너시스비비큐는 2011년 9월부터 지난해 7월까지 가맹점이 소비자로부터 받은 상품권을 정산하면서 액면가의 10%를 발행수수료로 공제했다. 제너시스비비큐는 또 높은 수수료 부담을 피하기 위해 고객들로부터 상품권을 받지 않은 가맹점에는 가맹계약을 해지할 수 있다는 증명을 발송해 상품권 수령을 강요하기도 했다. 이를 통해 본사가 챙긴 상품권 수수료는 2020만원이었다. 제너시스비비큐는 2012년 8월 이후에도 상품권 수수료 10%를 공제해오다 지난 6월부터 수수료율을 3%로 낮췄다. 공정위는 "가맹본부가 포인트나 상품권 비용을 동의나 정당한 근거없이 가맹점주에게 전가한 행위를 법 위반으로 조치함으로써 앞으로 유사한 사례가 재발되지 않도록 하는 효과가 있을 것으로 기대한다"고 밝혔다.'

In [25]:
total_questions[0]

'제너시스비비큐의 대표이사는 누구야?'

In [26]:
no_answer_df = pd.DataFrame({'contexts': total_contexts, 'questions': total_questions, 'group_id': group_numbers})
no_answer_df

,contexts,questions,group_id
0,공정거래위원회는 상품권 발행비용 명목으로 고율의 발행수수료를 가맹점주에게 부담토록 ...,제너시스비비큐의 대표이사는 누구야?,2
1,공정거래위원회는 상품권 발행비용 명목으로 고율의 발행수수료를 가맹점주에게 부담토록 ...,제너시스 비비큐의 창립일은 언제야?,2
2,공정거래위원회는 상품권 발행비용 명목으로 고율의 발행수수료를 가맹점주에게 부담토록 ...,공정거래위원회는 어디에 사무실이 위치해있어?,2
3,공정거래위원회는 상품권 발행비용 명목으로 고율의 발행수수료를 가맹점주에게 부담토록 ...,공정거래위원회가 제너시스비비큐에게 내린 시정명령은 어떤 내용이야?,2
4,공정거래위원회는 상품권 발행비용 명목으로 고율의 발행수수료를 가맹점주에게 부담토록 ...,다른 회사는 상품권 발행비용 명목으로 수수료를 얼마씩 받고있어?,2
...,...,...,...
100239,"""남북 정상회담을 성공으로 이끌기 위한 환영만찬의 메뉴가 공개됐습니다. 또 오차 없...",정상회담을 앞두고 예행 연습을 진행하는 이유는 뭐야?,1
100240,"""남북 정상회담을 성공으로 이끌기 위한 환영만찬의 메뉴가 공개됐습니다. 또 오차 없...",양 정상의 이동 경로와 의전 등은 어떻게 점검해?,1
100241,"""북미정상회담 준비가 잘 진척되고 있다고 미국 유력 언론이 잇따라 보도하고 있습니다...",북미정상회담은 언제 하나?,1
100242,"""북미정상회담 준비가 잘 진척되고 있다고 미국 유력 언론이 잇따라 보도하고 있습니다...",북미정상회담은 어디에서 하나?,1


In [27]:
# 본문 중복 제거
no_answer_df = no_answer_df.drop_duplicates(subset='contexts')

In [28]:
system_prompt = """당신은 본문으로부터 질문의 답변을 작성하는 언어 모델입니다.
본문에 질문의 답이 없는 경우 질문 내용을 언급하면서 해당 내용은 답을 찾을 수 없다고 답변하고 도움이 더 필요하면 고객센터로 연락하라고 안내하세요.

고객센터의 전화번호는 02-1234-5678입니다."""

user_prompt = []
for c, q in zip(no_answer_df['contexts'].to_list(), no_answer_df['questions'].to_list()):
  user_prompt.append('본문: ' + c + '\n' + '질문: ' + q + '\n답변:')

In [29]:
# gpt에게 질문에 대한 답을 응답 받기
response = client.chat.completions.create(
    model = 'gpt-4o',
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt[0]}
    ]
)

In [30]:
response.choices[0].message.content

'본문에는 제너시스비비큐의 대표이사에 대한 정보가 없습니다. 추가적인 도움이 필요하시면 고객센터 02-1234-5678로 연락하시기 바랍니다.'

이런식으로 Positive Sampling, Negative Sampling 데이터 세트를 많이 만들어 낼 수 있다. -> RAFT

# 학습할 모델의 토큰 개수 확인
RAG 구축을 위해 이 모델이 최대 몇 개의 토큰을 받을 수 있는지를 확인해야 합니다. Open LLM(허깅페이스)를 사용하는 경우 `File and Versions` 탭의 `config.json`에서 `max_position_embeddings` 값을 찾아보면 됩니다. `allganize/Llama-3-Alpha-Ko-8B-Evo`의 경우 최대 토큰의 개수는 8192개가 됩니다.

주의해야 할 점은, 8192개로 최대 토큰이 설정된 모델에 7000개 가량의 토큰을 집어 넣게 되면 답변은 `8192-7000=1192`개 까지 받게 됩니다.

따라서 각 본문들이 평균적으로 길이가 1000이고, 받아야 할 답변도 마찬가지로 길이가 1000이라면 대략적으로 질문에 대한 본문의 유사도 top-7 까지는 넣을 수 있겠다고 생각할 수 있겠습니다.

In [31]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('allganize/Llama-3-Alpha-Ko-8B-Evo')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

In [32]:
# 실제 검색에 사용될 문서를 넣고 대략적으로 토큰이 얼마정도 소모되는지 확인
# 실제 학습 시 사용할 프롬프트 예시
system_prompt = '''당신은 본문으로부터 질문의 답변을 작성하는 언어 모델입니다.

### 지시사항
1. 질문으로부터 검색 결과에서 답을 찾아 작성하세요.
2. 검색 결과에 질문에 대한 답이 없는 경우에는 답을 찾을 수 없다고 작성하세요. 답이 없다고 하고 고객 센터 번호를 안내하세요.
3. source_id는 문서 번호입니다. 따라서 답변을 하는 경우 몇 번 문서를 인용했는지 답변 뒤에 언급하세요.

검색 결과:
["Source_id": doc1,"Content": "노키즈존 확산, 어떻게 볼 것인가?이슈 & 진단4<노키즈존 정보제공 사이트 1><노키즈존 정보제공 사이트 2>자료 :Leavethembehind.com3.자료 :Travelandleisure.com.○어린이와 일반 고객의 탑승 공간을 구분하는 항공사도 확산 추세-2012년 말레이시아 항공은 12세 이하의 아이와 동승자는 항공기 아래층 지정구역에만 착석할 수 있도록 하는 정책 도입-에어아시아 엑스는 항공기 내에 ‘콰이엇 존(quietzone)’을 설치하고 있으며,스쿠트 항공은 2013년부터 아이들을 위한 전용좌석제 실시○영국예약사이트 ‘레이트딜’에서 비행이용승객 1,108명을 대상으로 설문조사 실시-응답자의 70%가 비행기내 노키즈존 도입을 찬성하였으며,그 중 35%는 노키즈존에 앉기 위해 추가요금도 지불할 수 있다고 응답<에어아시아의 ‘콰이엇 존’><비행 중 싫어하는 행동:앞좌석 발차기>자료 :AirAsia사이트(http://www.airasia.com/kr/ko/home.page).자료 :googleimage(https://www.google.co.kr)."]
["Source_id": doc2,"Content": "노키즈존 확산, 어떻게 볼 것인가?이슈 & 진단3 노키즈존은 다른 나라에서도 확산 추세영국에서는 어린이의 펍(Pub)출입을 둘러싼 논쟁이 지속되는 상황○영국은 1995년에 부모가 아이를 동반할 경우 아이도 펍에 출입할 수 있도록 법률 개정-법률 개정 전에는 만 14세 미만은 펍 출입이 허용되지 않았음○법률 개정 후 ‘부모를 동반한 어린이의 펍 출입’이 논란이 되면서 BBC는 2010년 시청자 토론방 개설-“소란스럽고 우는 아이들 때문에 분위기를 즐길 수 없다”또는 “펍은 어른들의 전유물로 남겨둬야 한다”는 의견이 압도적미국에서는 노키즈존이 민권법(CivilRightsAct)과 충돌하는지가 논란○미국에서도 어린아이를 동반한 고객들이 증가하면서 노키즈존을 도입하는 레스토랑 증가 추세○하지만 어린아이의 출입을 제한하는 것은 차별을 엄격히 금지하는 민권법에 위배된다는 주장 제기-미국 민권법은 인종이나 종교 등에 따른 차별을 엄격하게 금지하고 있는데,이에 근거해 볼 때 노키즈존도 위법하다는 주장노키즈존 정보를 알려주는 여행 사이트나 항공사도 등장○여행 사이트나 블로그를 통해 어린이 관련 정보를 제공하는 업체 증가-영국의 여행 사이트 ‘Leavethembehind.com’은 아이들의 방해를 받지 않고 휴가를 즐길 수 있는 여행 정보 제공-여행정보제공 블로그 ‘TravelandLeisures’는 아이들의 출입을 제한하는 레스토랑 리스트 제공"]
["Source_id": doc3,"Content": "노키즈존 확산, 어떻게 볼 것인가?이슈 & 진단7Ⅱ.노키즈존 관련 쟁점 노키즈존은 업주의 영업상의 자유에 해당하는가?어린이 고객의 출입을 제한하는 영업방침은 정부의 규제 대상이 아니라는 견해○대한민국 헌법 제 15조는 직업행사의 자유를 보장하고 있으며,이에 근거할 때 업주의 영업방침은 업주의 고유한 기본권에 해당-게다가 영업공간에서 사고가 발생했을 때 손해배상의 책임은 업주에게 귀속되기 때문에 영업방침을 규제하는 것은 과도한 재산권 침해에 해당○택시와 같이 공익성이 인정되는 서비스의 경우에는 정부의 개입이 인정되지만 카페나 음식점은 공익성이 인정되지 않는 사례에 해당-가령,택시 서비스는 공익성이 인정되고 소비자의 선택도 제한되기 때문에 법률에 의해 여객의 승차를 거부하는 행위를 금지-반면,카페나 음식점은 공익성이 인정되지도 않을 뿐만 아니라 소비자의 입장에서도 선택이 가능하기 때문에 정부의 개입은 과도한 규제에 해당노키즈존은 일반불공정 거래의 거래거절과 차별적 취급에도 해당되지 않는 사례○일부 전문가들은 노키즈존이 일반불공정 거래의 거래거절과 차별적 취급에 해당한다고 주장-공정거래법 제23조 1항 1호는 ‘부당하게 거래를 거절하거나 거래의 상대방을 차별하여 취급하는 행위’를 금지하는데,노키즈존은 정당한 이유 없는 거래거절과 차별적 취급에 해당"]
["Source_id": doc4,"Content": "노키즈존 확산, 어떻게 볼 것인가?이슈 & 진단1Ⅰ.노키즈존 확산 특정 장소에 아이의 출입을 금지하는 노키즈존 확산 추세아이를 동반한 고객을 받지 않겠다고 표방하는 카페나 음식점 증가○노키즈존(NoKidsZone)이란 ‘아이를 동반하고 입장할 수 없는 공간’을 의미하는 것으로 아이들의 특정 행동이나 소음을 막기 위한 조치로 확산-강남이나 홍대 등 상업지구의 카페나 음식점에서 시작해서 다른 지역으로 확산-경기도의 경우 최근 수원시나 성남시,고양시 등 어린이들이 많이 거주하는 지역을 중심으로 확산<노키즈존 시행 매장 1><노키즈존 시행 매장 2>자료 :googleimage(https://www.google.co.kr).○아이의 소란스런 행동과 부모의 방관이 노키즈존 확산의 주된 원인-카페나 음식점 등 공공장소에서 소리 지르거나 뛰어노는 아이들,이를 방치하는 부모들이 노키즈존 확산의 주된 원인으로 지목-최근에는 카페나 음식점 내에서 기저귀를 갈고 그대로 두고 간다던지,컵으로 아이의 소변을 받는 등 일부 부모의 경우 없는 행동이 논란 야기"]
["Source_id": doc5,"Content": "노키즈존 확산, 어떻게 볼 것인가?이슈 & 진단6노키즈존을 알고 있거나 들어본 적이 있는 응답자도 대다수○노키즈존에 대해 들어본 적이 있다고 응답한 비율은 71%로 아이들로 인해 불편을 경험한 적이 있다는 응답보다는 적지만 상당히 높은 편-성별로는 여성의 78.5%,남성의 48.4%가 노키즈존을 알고 있다고 응답했으며,자녀 유무별로는 만 10세 미만 자녀가 있는 경우 82.2%,만 10세 미만 자녀가 없는 경우 59.8%가 노키즈존을 알고 있다고 응답<전체><성별><10세 미만 자녀유무>자료 :경기연구원 모바일 설문조사(2016).○몇몇 설문조사 결과 자녀를 둔 엄마를 포함 대부분의 시민들은 노키즈존에 대해 찬성 의견 표명-2014년 9월,엄마들이 회원인 육아카페 ‘맘스홀릭베이비’에서 회원 3,525명을 대상으로 노키즈존 찬반 관련 설문을 실시한 결과 찬성이 72.7%로 압도적-2015년 9월,JTBC‘뉴스룸’에서 길거리 시민들을 대상으로 노키즈존 찬반 의견을 물은 결과도 찬성이 63%로 우세○‘알바몬’에서 아르바이트생 1,084명에게 ‘근무 중인 매장이 노키즈존으로 변경된다면 찬성할 것이냐’고 물은 결과 찬성의견이 65.5%에 달함-근무 도중 ‘유아 또는 유아를 동반한 고객으로 인해 곤란을 겪은 적이 있다’고 응답한 아르바이트생이 67.7%로,유아 고객으로 인한 업무부담 증가가 노키즈존 찬성의 주된 이유"]

질문: 노키즈존을 도입한 항공사는 어디인가요?

답변:'''

# 예상 답변.. 이거도 gpt-4o 같은 모델로 만들 수 있음
answer = '''노키즈존을 도입한 항공사는 몇 군데가 있습니다. 예를 들어, 2012년 말레이시아 항공은 12세 이하의 아이와 동승하는 고객들에게 항공기 아래층 지정구역에만 착석할 수 있도록 하는 정책을 도입했습니다. 또 다른 예로는 에어아시아 엑스가 있습니다. 이 항공사는 항공기 내에 '콰이엇 존(quietzone)'을 설치하여 아이들이 좀 더 조용한 환경에서 여행할 수 있도록 했습니다. 그리고 스쿠트 항공은 2013년부터 아이들을 위한 전용좌석제를 실시하고 있습니다.[[doc1]][[doc2]][[doc3]]'''

In [33]:
# 시스템 프롬프트 토큰 수 + 답변 토큰수 확인. 얼마나 토큰의 여유가 있을지를 확인
len(tokenizer.tokenize(system_prompt + answer))

2268

In [34]:
8192-2268

5924

# 질문과 유사도가 높은 문서 기반의 데이터 구축
실제로는 사용자들이 입력할만한 질문을 넣어야 하기 때문에 Human Labeling이 필요할 수도 있으나, 최근에는 질문 자체를 뛰어난 LLM 모델( 예를 들면 gpt-4o-turbo, gpt-4o 등 )을 기반으로 만드는 것도 꽤나 괜찮습니다. 문서를 주고, 해당 문서로 답할 수 있는 질문을 만들어 달라고 하면 됩니다.

In [35]:
result_df['contexts'].loc[0]

"한국청소년단체협의회와 여성가족부는 22일부터 28일까지 서울과 충북 괴산에서 '국제청소년포럼'을 연다고 21일 밝혔다. 한국 미국 캐나다 호주 등 전 세계 32개국 75여명의 대학생, 청소년들이 모여 전 세계적 현안문제에 대한 대안과 해결책을 모색하는 자리다. 이번 포럼의 주제는 '청소년과 뉴미디어'다. 스마트폰 SNS 태블릿PC 등 새로운 커뮤니케이션 매체인 '뉴미디어'에 대한 성찰과 문제점에 대해 토론한다. 기조강연을 시작으로 국가별 주제관련 사례발표, 그룹 토론 및 전체총회, '청소년선언문' 작성 및 채택 등 다양한 프로그램을 운영한다. 개회식은 22일 서울 방화동에 있는 국제청소년센터 국제회의장에서 한다. 전 세계 32개국 대학생ㆍ청소년 참가자와 전국의 청소년기관단체장과 청소년지도자 여성가족부 주한외교사절 등 100여명이 참석할 예정이다. 23일에는 유엔미래포럼 박영숙 대표가 '뉴미디어의 균형 있는 발전을 위한 청소년의 역할'에 대해 기조강연을 한다. 뉴미디어의 올바른 활용방안과 청소년문화의 형성에 대해 설명할 계획이다. 27일 폐회식에서는 '청소년선언문'을 채택한다. 선언문에는 전 세계적으로 뉴미디어의 바람직한 발전을 촉구하며 각국 청년들이 함께 실천할 수 있는 내용 등이 담길 예정이다. 한국청소년단체협의회는 포럼이 끝난 뒤 UN 등 국제기구와 참가자 각국 정부 등 국제사회에 선언문을 전달할 예정이다."

In [36]:
# 본문으로부터 질문을 만들어 내기 위한 프롬프트 작성
system_prompt = """주어진 문서를 이용하여 답변할 수 있는 간단한 질문 5개를 파이썬 문자열 리스트 형태로 제안하세요."""

user_prompt = []
for doc in result_df['contexts'].to_list():
  user_prompt.append('문서: ' + doc + '\n질문 5개를 만드세요.\n시작!\n질문 리스트:')

In [37]:
print(user_prompt[0])

문서: 한국청소년단체협의회와 여성가족부는 22일부터 28일까지 서울과 충북 괴산에서 '국제청소년포럼'을 연다고 21일 밝혔다. 한국 미국 캐나다 호주 등 전 세계 32개국 75여명의 대학생, 청소년들이 모여 전 세계적 현안문제에 대한 대안과 해결책을 모색하는 자리다. 이번 포럼의 주제는 '청소년과 뉴미디어'다. 스마트폰 SNS 태블릿PC 등 새로운 커뮤니케이션 매체인 '뉴미디어'에 대한 성찰과 문제점에 대해 토론한다. 기조강연을 시작으로 국가별 주제관련 사례발표, 그룹 토론 및 전체총회, '청소년선언문' 작성 및 채택 등 다양한 프로그램을 운영한다. 개회식은 22일 서울 방화동에 있는 국제청소년센터 국제회의장에서 한다. 전 세계 32개국 대학생ㆍ청소년 참가자와 전국의 청소년기관단체장과 청소년지도자 여성가족부 주한외교사절 등 100여명이 참석할 예정이다. 23일에는 유엔미래포럼 박영숙 대표가 '뉴미디어의 균형 있는 발전을 위한 청소년의 역할'에 대해 기조강연을 한다. 뉴미디어의 올바른 활용방안과 청소년문화의 형성에 대해 설명할 계획이다. 27일 폐회식에서는 '청소년선언문'을 채택한다. 선언문에는 전 세계적으로 뉴미디어의 바람직한 발전을 촉구하며 각국 청년들이 함께 실천할 수 있는 내용 등이 담길 예정이다. 한국청소년단체협의회는 포럼이 끝난 뒤 UN 등 국제기구와 참가자 각국 정부 등 국제사회에 선언문을 전달할 예정이다.
질문 5개를 만드세요.
시작!
질문 리스트:


In [38]:
# gpt에게 질문에 대한 답을 응답 받기
response = client.chat.completions.create(
    model = 'gpt-4o',
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt[0]}
    ]
)

In [39]:
print(response.choices[0].message.content)

```python
questions = [
    "국제청소년포럼은 언제부터 언제까지 열리나요?",
    "이번 포럼의 주요 주제는 무엇인가요?",
    "포럼에서 어떤 프로그램들이 운영되나요?",
    "개회식은 어디에서 열리나요?",
    "폐회식에서는 어떤 중요한 문서가 채택될 예정인가요?"
]
```


In [40]:
!pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 255.2/255.2 kB 7.5 MB/s eta 0:00:00


In [41]:
from sentence_transformers import SentenceTransformer

sentence_vectorizer = SentenceTransformer("BAAI/bge-m3")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [42]:
tqdm.pandas()

In [43]:
question = "서울과 충북 괴산에서 '국제청소년포럼'을 여는 곳은?"

In [44]:
vec = sentence_vectorizer.encode(question)
print(list(vec))

[0.016701674, -0.025745831, -0.026942484, -0.038272224, -0.023840012, -0.022720383, -0.004105383, -0.024989014, -0.037059207, 0.0047121714, 0.02178352, 0.06450989, -0.018987684, 0.005559755, 0.048679247, 0.010542924, -0.019103909, 0.009894576, -0.030277498, -0.042012095, -0.041594435, 0.0059922347, -0.033741146, -0.03971184, 0.05005681, -0.016873194, -0.0028756913, -0.012966876, -0.002207693, -0.02178685, -0.02349155, 0.006065555, 0.028758144, -0.06582626, -0.016454848, -0.039071076, 0.0019614012, -0.027612053, -0.06820398, -0.043937508, 0.005100627, -0.00369064, -0.008331585, 0.04530395, -0.022295728, -0.030338487, 0.026355881, -0.016214471, 0.017712833, -0.0027515816, -0.017881628, -0.009192591, -0.036201708, -0.025410207, 0.0021700717, 0.029742815, 0.027743852, 0.01252651, -0.046879735, -0.034629006, -0.031610638, -0.003851233, -0.0073858835, 0.024750723, 0.0021772902, 0.0030534987, 0.008974831, -0.006075663, -0.0136490865, -0.0019990022, -0.034653947, -0.009882593, 0.046684008, -0.

In [45]:
# 본문에 대한 임베딩 벡터 얻기
result_df['document_embedding'] = result_df.progress_apply(lambda row : sentence_vectorizer.encode(row.contexts), axis=1)

100%|██████████| 4500/4500 [09:01<00:00,  8.32it/s]


In [47]:
result_df.head()

,contexts,questions,answers,group_id,document_embedding
0,한국청소년단체협의회와 여성가족부는 22일부터 28일까지 서울과 충북 괴산에서 '국제...,서울과 충북 괴산에서 '국제청소년포럼'을 여는 곳은?,"{'answer_start': 0, 'text': '한국청소년단체협의회와 여성가족부...",5,"[0.015234392, -0.030399576, -0.027278177, -0.0..."
1,◈ NASA의 화성 탐사 로버인 오퍼튜니티(Opportunity)가 화성 표면에서 ...,화성 탐사 로버인 오퍼튜니티는 누가 만든거야?,"{'answer_start': 2, 'text': 'NASA', 'answer_en...",5,"[0.002617271, 0.020062923, -0.060203325, 0.054..."
2,민주통합당 이해찬 대표는 1일 검찰의 박지원 원내대표에 대한 수사 중단과 부자감세 ...,1일 검찰의 박지원 원내대표에 대한 수사 중단과 부자감세 철회 등을 주장한 사람은 ...,"{'answer_start': 0, 'text': '민주통합당 이해찬 대표', 'a...",5,"[-0.027281206, 0.011722905, -0.008380434, -0.0..."
3,"""오늘 전국이 25도 안팎의 날씨를 보였죠. 기온이 26도 정도가 되면 맥주의 매출...",5년 전 93억원 규모였던 시장이 지난해 400억원 규모로 커졌고 5년 뒤에는 15...,"{'answer_start': 768, 'text': '수제맥주협회', 'answe...",5,"[-0.0015891646, 0.04678919, -0.027548613, 0.00..."
4,극심한 구인난을 겪고 있는 일본 기업들에는 어렵게 뽑은 인력이 잠깐 일하다 그만두고...,신입사원 채용 후 세심한 관리로 이직 리스크를 낮추는 서비스를 제공하는 정보기술 업체는?,"{'answer_start': 488, 'text': '엔재팬', 'answer_e...",5,"[-0.10692061, -0.019807039, -0.028451355, -0.0..."


In [48]:
import numpy as np
from numpy import dot
from numpy.linalg import norm

def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

# 질문을 입력받아 해당 질문의 벡터를 구한 후 본문의 유사도벡터(document_embedding)와의 유사도를 계산하기 위한 함수
def return_answer_candidate(df, query, n=5):
    query_embedding = sentence_vectorizer.encode(query)
    df["similarity"] = df.document_embedding.apply(lambda x: cos_sim(np.array(x), np.array(query_embedding)))
    top_n_docs = df.sort_values("similarity", ascending=False).head(n)
    return top_n_docs

In [49]:
# orion 식(CoT)으로 표현하기를 강제합니다. orion은 문서 인용을 거의 강제하는 모델로서, 프롬프트 엔지니어링을 통해 llama나 gpt를 사용할 때도 orion 표현을 따르게 하는 것이 가능합니다.
system_prompt = """당신은 검색된 문서부터 질문의 답변을 작성하는 언어 모델입니다.

### 지시사항
1. 질문으로부터 주어진 다수의 문서에서 답을 찾아 작성하세요.
2. 검색된 문서에 질문의 답이 없는 경우에는 질문에 대한 내용을 언급하면서 답을 찾을 수 없다고 작성하세요.
3. 답변에서 참고자료의 내용을 인용한 경우, 답변 맨 끝에 인용한 Source_id를 추가하세요. 답변에 내용을 인용한 경우에만 작성해야 합니다. 인용한 게 없다면 쓰지마세요.
4. Source_id 값은 검색 결과에서 가져온 것이며 두 개의 대괄호로 묶어야 합니다. 예: [[d97b811489b73f46c8d2cb1bc888dbbe]], [[b6be48868de736b90363d001c092c019]]"""

In [46]:
question = result_df['questions'][0]
question

"서울과 충북 괴산에서 '국제청소년포럼'을 여는 곳은?"

In [50]:
# 질문과 유사한 본문 찾기
sample_df = return_answer_candidate(result_df, question)
sample_df

,contexts,questions,answers,group_id,document_embedding,similarity
0,한국청소년단체협의회와 여성가족부는 22일부터 28일까지 서울과 충북 괴산에서 '국제...,서울과 충북 괴산에서 '국제청소년포럼'을 여는 곳은?,"{'answer_start': 0, 'text': '한국청소년단체협의회와 여성가족부...",5,"[0.015234392, -0.030399576, -0.027278177, -0.0...",0.629088
4464,청소년 지킴이 강지원 변호사와 진보적 문학평론가 임헌영(중앙대 교수) 민족문제연구소...,민족문제연구소장은 누구인가요?,"{'text': '임헌영', 'answer_start': 27, 'answer_en...",8,"[-0.029915512, 0.0084119905, -0.04014301, -0.0...",0.435533
74,포스코건설(대표이사 이영훈)은 충청북도 청주시에서 처음으로 선보이는 ‘더샵’아파트인...,포스코건설의 대표이사는 누구인가?,"{'text': '이영훈', 'answer_start': 11, 'answer_en...",5,"[-0.03605716, 0.0056840256, -0.032754466, -0.0...",0.425657
792,"갈 수 없는, 하지만 가보고 싶은 땅. 한반도의 허리를 끊고 이별로 목 놓아 부른 ...",경기도교육청의 지원으로 금강산을 2박3일로 다녀온 사람은 누구야?,"{'text': '400명이 넘는 경기도 내 고등학교 학생들', 'answer_st...",7,"[0.033612154, 0.015714975, -0.058121674, -0.04...",0.424542
749,"""북한의 원산-금강산 관광지구 개발 계획안입니다. 북한은 3년 전 김정은의 지시로 ...",북한의 원산-금강산 관광지구 개발 계획안은 누구의 지시로 시작됐나?,"{'text': '김정은', 'answer_start': 37, 'answer_en...",7,"[0.012351787, -0.014326906, -0.08282027, -0.02...",0.421799


In [51]:
# 직접 사용자가 넣을만한 질문으로 테스트
question = "이명박 대통령 소식"

sample_df = return_answer_candidate(result_df, question)
sample_df

,contexts,questions,answers,group_id,document_embedding,similarity
3405,"""검찰이 이명박 전 대통령에게 오는 14일 출석하라고 통보했습니다. 100억 원에 ...",누가 이명박 전 대통령에게 오는 14일 출석하라고 통보했나?,"{'text': '검찰', 'answer_start': 1, 'answer_end'...",9,"[0.021000123, 0.008971538, -0.023830399, 0.026...",0.548822
3817,"""이명박 전 대통령은 구속 이후 처음으로 검찰 수사 결과에 대해 입장을 밝혔습니다....",이명박 전 대통령은 다스는 누구의 회사라고 했는가?,"{'answer_start': 390, 'text': '이상은 회장의 가족 회사',...",1,"[-0.03805591, -0.050939515, -0.018412163, 0.00...",0.541001
3698,"""이명박 전 대통령이 소환된 지 21시간 만에 귀가했습니다. 이 전 대통령은 검찰 ...",검찰 조사에서 자신의 혐의를 대부분 부인하고 소환된 지 21시간 만에 귀가한 사람은?,"{'answer_start': 1, 'text': '이명박 전 대통령', 'answ...",1,"[-0.005722005, -0.0034464023, -0.0035507868, 0...",0.527119
3513,"""이명박 전 대통령이 소환된 지 21시간 만에 귀가했습니다. 이 전 대통령은 검찰 ...",이명박 전 대통령은 검찰에 소환된 지 몇시간 만에 귀가했는가?,"{'text': '21시간', 'answer_start': 18, 'answer_e...",1,"[-0.0034160444, -0.008739646, -0.0050890693, 0...",0.526913
3638,이명박 대통령이 검찰의 저축은행 비리 수사를 질타한 것으로 확인됐다. 청와대가 검찰...,특검 여부는 누가 결정할 문제야?,"{'text': '국회', 'answer_start': 521, 'answer_en...",1,"[0.004959098, -0.007008047, -0.036336627, -0.0...",0.519629


이어서 인용할 문서의 id(Source_id)를 생성합니다. 질문에 대한 문서의 유사도 top-5를 모델이 참고할 문서로 사용합니다.

In [54]:
search_lst = []

for i, c in zip(sample_df.index.to_list(), sample_df['contexts'].to_list()):
  search_lst.append(["Source_id" + ": " + str(i), "Content" + ": " + str(c)])

search_lst[:3]

[['Source_id: 3405',
  'Content: "검찰이 이명박 전 대통령에게 오는 14일 출석하라고 통보했습니다. 100억 원에 이르는 불법자금 수수 등의 혐의를 받는 피의자 신분입니다. 오대성 기자입니다. [리포트] 14일 오전 9시 반. 이명박 전 대통령이 검찰에 소환됩니다. 불법자금 수수 등의 혐의를 받는 피의자 신분으로, 검찰 수사 착수 약 150일 만입니다. 소환의 필요성에 대해 검찰 관계자는 실체적 진실을 효율적이고 투명하게 밝히기 위해서라고 말했습니다. 조사 장소는 서울중앙지검 10층에 있는 특별조사실이 유력하게 거론되고 있습니다. 1년 전 박근혜 전 대통령이 조사를 받았던 곳입니다. 전직 대통령에 대한 예우도 박 전 대통령과 비슷한 수준이 될 전망입니다. 조사를 받기 전 윤석열 서울중앙지검장이 이 전 대통령을 응대합니다. 조사 방식 등에 대한 설명은 수사 실무책임자인 한동훈 3차장 검사가 진행합니다. 조사에는 특수2부와 첨단범죄수사1부 부장검사가 투입될 예정입니다. 검찰은 전직 대통령이라는 점을 고려해 한 차례 소환으로 조사를 끝낼 방침입니다. 혐의를 전면 부인하고 있는 이 전 대통령 측은 &소환에는 응하지만 날짜는 협의하겠다&고 밝혔습니다. 지금까지 드러난 범죄 혐의를 봤을때 소환 조사 다음 수순은 구속영장 청굽니다. 이 전 대통령 형 이상득 전 의원은 오늘 검찰에 재소환됩니다. 이 전 의원은 이명박정부 때 국가정보원으로부터 특수활동비를 뇌물로 받은 혐의와 각종 불법 자금을 받은 혐의 등의 피의자 신분입니다.검찰은 지난 1월 이 전 의원을 소환했지만 건강상의 이유로 4시간 가량 조사한 뒤 귀가조치 했습니다. KBS 뉴스 오대성입니다."'],
 ['Source_id: 3817',
  'Content: "이명박 전 대통령은 구속 이후 처음으로 검찰 수사 결과에 대해 입장을 밝혔습니다. 정권의 하수인이 짜맞춘 표적 수사라고 맹비난했습니다. 혐의도 조목조목 반박했습니다. 이세연 기자의 보도입니다. [리포트] 이명박 전 대통령 SNS에 글이

In [55]:
# 내용을 문자열로 이어줍니다.
search_result = ''
for s in search_lst:
  search_result += str(s) + '\n'

print(search_result)

['Source_id: 3405', 'Content: "검찰이 이명박 전 대통령에게 오는 14일 출석하라고 통보했습니다. 100억 원에 이르는 불법자금 수수 등의 혐의를 받는 피의자 신분입니다. 오대성 기자입니다. [리포트] 14일 오전 9시 반. 이명박 전 대통령이 검찰에 소환됩니다. 불법자금 수수 등의 혐의를 받는 피의자 신분으로, 검찰 수사 착수 약 150일 만입니다. 소환의 필요성에 대해 검찰 관계자는 실체적 진실을 효율적이고 투명하게 밝히기 위해서라고 말했습니다. 조사 장소는 서울중앙지검 10층에 있는 특별조사실이 유력하게 거론되고 있습니다. 1년 전 박근혜 전 대통령이 조사를 받았던 곳입니다. 전직 대통령에 대한 예우도 박 전 대통령과 비슷한 수준이 될 전망입니다. 조사를 받기 전 윤석열 서울중앙지검장이 이 전 대통령을 응대합니다. 조사 방식 등에 대한 설명은 수사 실무책임자인 한동훈 3차장 검사가 진행합니다. 조사에는 특수2부와 첨단범죄수사1부 부장검사가 투입될 예정입니다. 검찰은 전직 대통령이라는 점을 고려해 한 차례 소환으로 조사를 끝낼 방침입니다. 혐의를 전면 부인하고 있는 이 전 대통령 측은 &소환에는 응하지만 날짜는 협의하겠다&고 밝혔습니다. 지금까지 드러난 범죄 혐의를 봤을때 소환 조사 다음 수순은 구속영장 청굽니다. 이 전 대통령 형 이상득 전 의원은 오늘 검찰에 재소환됩니다. 이 전 의원은 이명박정부 때 국가정보원으로부터 특수활동비를 뇌물로 받은 혐의와 각종 불법 자금을 받은 혐의 등의 피의자 신분입니다.검찰은 지난 1월 이 전 의원을 소환했지만 건강상의 이유로 4시간 가량 조사한 뒤 귀가조치 했습니다. KBS 뉴스 오대성입니다."']
['Source_id: 3817', 'Content: "이명박 전 대통령은 구속 이후 처음으로 검찰 수사 결과에 대해 입장을 밝혔습니다. 정권의 하수인이 짜맞춘 표적 수사라고 맹비난했습니다. 혐의도 조목조목 반박했습니다. 이세연 기자의 보도입니다. [리포트] 이명박 전 대통령 SNS에 글이 하나 게시됐

In [56]:
user_prompt = f"""검색 결과:
{search_result}
질문: {question}
답변:
"""

print(user_prompt)

검색 결과:
['Source_id: 3405', 'Content: "검찰이 이명박 전 대통령에게 오는 14일 출석하라고 통보했습니다. 100억 원에 이르는 불법자금 수수 등의 혐의를 받는 피의자 신분입니다. 오대성 기자입니다. [리포트] 14일 오전 9시 반. 이명박 전 대통령이 검찰에 소환됩니다. 불법자금 수수 등의 혐의를 받는 피의자 신분으로, 검찰 수사 착수 약 150일 만입니다. 소환의 필요성에 대해 검찰 관계자는 실체적 진실을 효율적이고 투명하게 밝히기 위해서라고 말했습니다. 조사 장소는 서울중앙지검 10층에 있는 특별조사실이 유력하게 거론되고 있습니다. 1년 전 박근혜 전 대통령이 조사를 받았던 곳입니다. 전직 대통령에 대한 예우도 박 전 대통령과 비슷한 수준이 될 전망입니다. 조사를 받기 전 윤석열 서울중앙지검장이 이 전 대통령을 응대합니다. 조사 방식 등에 대한 설명은 수사 실무책임자인 한동훈 3차장 검사가 진행합니다. 조사에는 특수2부와 첨단범죄수사1부 부장검사가 투입될 예정입니다. 검찰은 전직 대통령이라는 점을 고려해 한 차례 소환으로 조사를 끝낼 방침입니다. 혐의를 전면 부인하고 있는 이 전 대통령 측은 &소환에는 응하지만 날짜는 협의하겠다&고 밝혔습니다. 지금까지 드러난 범죄 혐의를 봤을때 소환 조사 다음 수순은 구속영장 청굽니다. 이 전 대통령 형 이상득 전 의원은 오늘 검찰에 재소환됩니다. 이 전 의원은 이명박정부 때 국가정보원으로부터 특수활동비를 뇌물로 받은 혐의와 각종 불법 자금을 받은 혐의 등의 피의자 신분입니다.검찰은 지난 1월 이 전 의원을 소환했지만 건강상의 이유로 4시간 가량 조사한 뒤 귀가조치 했습니다. KBS 뉴스 오대성입니다."']
['Source_id: 3817', 'Content: "이명박 전 대통령은 구속 이후 처음으로 검찰 수사 결과에 대해 입장을 밝혔습니다. 정권의 하수인이 짜맞춘 표적 수사라고 맹비난했습니다. 혐의도 조목조목 반박했습니다. 이세연 기자의 보도입니다. [리포트] 이명박 전 대통령 SNS에 글이

In [57]:
# gpt에게 질문에 대한 답을 응답 받기
response = client.chat.completions.create(
    model = 'gpt-4o',
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
)

print(response.choices[0].message.content)

이명박 전 대통령에 대한 최근 소식으로는 다음과 같은 내용들이 있습니다. 

1. 이명박 전 대통령은 100억 원에 이르는 불법 자금 수수 등의 혐의를 받고 검찰에 소환되었습니다. 소환은 검찰 수사 착수 약 150일 만에 이루어졌으며, 검찰은 전직 대통령이라는 점을 고려해 한 차례 소환으로 조사를 끝낼 방침이라고 밝혔습니다[[3405]].

2. 검찰 조사 후 이명박 전 대통령은 자신의 혐의를 대부분 부인한 것으로 전해졌습니다. 본인은 전혀 모르는 일이며 설령 그런 일이 있어도 실무선에서 이루어진 일이라고 해명했습니다. 또한, 검찰이 압수한 증거물에 대해서 조작 가능성을 제기하기도 했습니다[[3698]], [[3513]].

3. 구속되기 전, 이명박 전 대통령은 자신의 혐의에 대해 검찰 수사가 "짜맞춘 표적 수사"라고 비난하며 혐의를 조목조목 반박했습니다. 다스 자금 횡령 혐의를 일축하며, 관련 혐의들을 부인하는 입장을 밝혔습니다. 또한, 국정원 특활비 상납에 대해서는 측근들에게 책임을 돌리고, 옥중조사를 거부했습니다[[3817]].


In [58]:
# LLaMA-3에 학습 시키고 싶다면? 다음과 같이 프롬프트를 만들면 됩니다.
prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

당신은 검색된 문서부터 질문의 답변을 작성하는 언어 모델입니다.

### 지시사항
1. 질문으로부터 주어진 다수의 문서에서 답을 찾아 작성하세요.
2. 검색된 문서에 질문의 답이 없는 경우에는 질문에 대한 내용을 언급하면서 답을 찾을 수 없다고 작성하세요.
3. 답변에서 참고자료의 내용을 인용한 경우, 답변 맨 끝에 인용한 Source_id를 추가하세요. 답변에 내용을 인용한 경우에만 작성해야 합니다. 인용한 게 없다면 쓰지마세요.
4. Source_id 값은 검색 결과에서 가져온 것이며 두 개의 대괄호로 묶어야 합니다. 예: [[d97b811489b73f46c8d2cb1bc888dbbe]], [[b6be48868de736b90363d001c092c019]]
<|eot_id|><|start_header_id|>user<|end_header_id|>
검색 결과:
["Source_id": doc1,"Content": "노키즈존 확산, 어떻게 볼 것인가?이슈 & 진단4<노키즈존 정보제공 사이트 1><노키즈존 정보제공 사이트 2>자료 :Leavethembehind.com3.자료 :Travelandleisure.com.○어린이와 일반 고객의 탑승 공간을 구분하는 항공사도 확산 추세-2012년 말레이시아 항공은 12세 이하의 아이와 동승자는 항공기 아래층 지정구역에만 착석할 수 있도록 하는 정책 도입-에어아시아 엑스는 항공기 내에 ‘콰이엇 존(quietzone)’을 설치하고 있으며,스쿠트 항공은 2013년부터 아이들을 위한 전용좌석제 실시○영국예약사이트 ‘레이트딜’에서 비행이용승객 1,108명을 대상으로 설문조사 실시-응답자의 70%가 비행기내 노키즈존 도입을 찬성하였으며,그 중 35%는 노키즈존에 앉기 위해 추가요금도 지불할 수 있다고 응답<에어아시아의 ‘콰이엇 존’><비행 중 싫어하는 행동:앞좌석 발차기>자료 :AirAsia사이트(http://www.airasia.com/kr/ko/home.page).자료 :googleimage(https://www.google.co.kr)."]
["Source_id": doc2,"Content": "노키즈존 확산, 어떻게 볼 것인가?이슈 & 진단3 노키즈존은 다른 나라에서도 확산 추세영국에서는 어린이의 펍(Pub)출입을 둘러싼 논쟁이 지속되는 상황○영국은 1995년에 부모가 아이를 동반할 경우 아이도 펍에 출입할 수 있도록 법률 개정-법률 개정 전에는 만 14세 미만은 펍 출입이 허용되지 않았음○법률 개정 후 ‘부모를 동반한 어린이의 펍 출입’이 논란이 되면서 BBC는 2010년 시청자 토론방 개설-“소란스럽고 우는 아이들 때문에 분위기를 즐길 수 없다”또는 “펍은 어른들의 전유물로 남겨둬야 한다”는 의견이 압도적미국에서는 노키즈존이 민권법(CivilRightsAct)과 충돌하는지가 논란○미국에서도 어린아이를 동반한 고객들이 증가하면서 노키즈존을 도입하는 레스토랑 증가 추세○하지만 어린아이의 출입을 제한하는 것은 차별을 엄격히 금지하는 민권법에 위배된다는 주장 제기-미국 민권법은 인종이나 종교 등에 따른 차별을 엄격하게 금지하고 있는데,이에 근거해 볼 때 노키즈존도 위법하다는 주장노키즈존 정보를 알려주는 여행 사이트나 항공사도 등장○여행 사이트나 블로그를 통해 어린이 관련 정보를 제공하는 업체 증가-영국의 여행 사이트 ‘Leavethembehind.com’은 아이들의 방해를 받지 않고 휴가를 즐길 수 있는 여행 정보 제공-여행정보제공 블로그 ‘TravelandLeisures’는 아이들의 출입을 제한하는 레스토랑 리스트 제공"]
["Source_id": doc3,"Content": "노키즈존 확산, 어떻게 볼 것인가?이슈 & 진단7Ⅱ.노키즈존 관련 쟁점 노키즈존은 업주의 영업상의 자유에 해당하는가?어린이 고객의 출입을 제한하는 영업방침은 정부의 규제 대상이 아니라는 견해○대한민국 헌법 제 15조는 직업행사의 자유를 보장하고 있으며,이에 근거할 때 업주의 영업방침은 업주의 고유한 기본권에 해당-게다가 영업공간에서 사고가 발생했을 때 손해배상의 책임은 업주에게 귀속되기 때문에 영업방침을 규제하는 것은 과도한 재산권 침해에 해당○택시와 같이 공익성이 인정되는 서비스의 경우에는 정부의 개입이 인정되지만 카페나 음식점은 공익성이 인정되지 않는 사례에 해당-가령,택시 서비스는 공익성이 인정되고 소비자의 선택도 제한되기 때문에 법률에 의해 여객의 승차를 거부하는 행위를 금지-반면,카페나 음식점은 공익성이 인정되지도 않을 뿐만 아니라 소비자의 입장에서도 선택이 가능하기 때문에 정부의 개입은 과도한 규제에 해당노키즈존은 일반불공정 거래의 거래거절과 차별적 취급에도 해당되지 않는 사례○일부 전문가들은 노키즈존이 일반불공정 거래의 거래거절과 차별적 취급에 해당한다고 주장-공정거래법 제23조 1항 1호는 ‘부당하게 거래를 거절하거나 거래의 상대방을 차별하여 취급하는 행위’를 금지하는데,노키즈존은 정당한 이유 없는 거래거절과 차별적 취급에 해당"]
["Source_id": doc4,"Content": "노키즈존 확산, 어떻게 볼 것인가?이슈 & 진단1Ⅰ.노키즈존 확산 특정 장소에 아이의 출입을 금지하는 노키즈존 확산 추세아이를 동반한 고객을 받지 않겠다고 표방하는 카페나 음식점 증가○노키즈존(NoKidsZone)이란 ‘아이를 동반하고 입장할 수 없는 공간’을 의미하는 것으로 아이들의 특정 행동이나 소음을 막기 위한 조치로 확산-강남이나 홍대 등 상업지구의 카페나 음식점에서 시작해서 다른 지역으로 확산-경기도의 경우 최근 수원시나 성남시,고양시 등 어린이들이 많이 거주하는 지역을 중심으로 확산<노키즈존 시행 매장 1><노키즈존 시행 매장 2>자료 :googleimage(https://www.google.co.kr).○아이의 소란스런 행동과 부모의 방관이 노키즈존 확산의 주된 원인-카페나 음식점 등 공공장소에서 소리 지르거나 뛰어노는 아이들,이를 방치하는 부모들이 노키즈존 확산의 주된 원인으로 지목-최근에는 카페나 음식점 내에서 기저귀를 갈고 그대로 두고 간다던지,컵으로 아이의 소변을 받는 등 일부 부모의 경우 없는 행동이 논란 야기"]
["Source_id": doc5,"Content": "노키즈존 확산, 어떻게 볼 것인가?이슈 & 진단6노키즈존을 알고 있거나 들어본 적이 있는 응답자도 대다수○노키즈존에 대해 들어본 적이 있다고 응답한 비율은 71%로 아이들로 인해 불편을 경험한 적이 있다는 응답보다는 적지만 상당히 높은 편-성별로는 여성의 78.5%,남성의 48.4%가 노키즈존을 알고 있다고 응답했으며,자녀 유무별로는 만 10세 미만 자녀가 있는 경우 82.2%,만 10세 미만 자녀가 없는 경우 59.8%가 노키즈존을 알고 있다고 응답<전체><성별><10세 미만 자녀유무>자료 :경기연구원 모바일 설문조사(2016).○몇몇 설문조사 결과 자녀를 둔 엄마를 포함 대부분의 시민들은 노키즈존에 대해 찬성 의견 표명-2014년 9월,엄마들이 회원인 육아카페 ‘맘스홀릭베이비’에서 회원 3,525명을 대상으로 노키즈존 찬반 관련 설문을 실시한 결과 찬성이 72.7%로 압도적-2015년 9월,JTBC‘뉴스룸’에서 길거리 시민들을 대상으로 노키즈존 찬반 의견을 물은 결과도 찬성이 63%로 우세○‘알바몬’에서 아르바이트생 1,084명에게 ‘근무 중인 매장이 노키즈존으로 변경된다면 찬성할 것이냐’고 물은 결과 찬성의견이 65.5%에 달함-근무 도중 ‘유아 또는 유아를 동반한 고객으로 인해 곤란을 겪은 적이 있다’고 응답한 아르바이트생이 67.7%로,유아 고객으로 인한 업무부담 증가가 노키즈존 찬성의 주된 이유"]
질문: 노키즈존을 도입한 항공사는 어디인가요?<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

# 문서가 긴 경우
위의 예시의 경우는 각 문서의 길이가 짧아서 서로 다른 문서들을 활용해 후보군을 만들 수 있었습니다. 하지만 만약 가지고 있는 문서가 너무 긴 경우(PDF나 스크래이핑한 데이터 등) 해당 문서를 분할하여 문서의 후보군을 만드는 것도 가능합니다.

In [59]:
import ast

In [60]:
!pip install langchain

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.4/404.4 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.0.0
    Uninstalling tenacity-9.0.0:
      Successfully uninstalled tenacity-9.0.0


In [63]:
import random
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [64]:
contexts = '''
사회투자정책과 재정관리


이재원;
초록

지식기반 경제체제로의 전환과 맞물려 지속 가능한 국가 성장동력을 위해서는 사회적 기반 확충이 중요하다. 따라서 사회투자정책은 적당한 수준에서 소비적 지출이 아닌 "성과에 책임지는 최적의 투자" 관점으로 접근되어야 한다. 이를 위해서는 사회개발과 경제개발의 균형있는 예산자원배분, 사회정책 부문간 균형있는 재원배분이 필요하다. 또한 프로그램 구조 전체를 전제로 하는 결과 지향적 성과관리와 납세자 책임 노력이 요구된다. 마지막으로 사회 복지재정 부담과 관련하여 중앙과 지방정부간 합리적 재정관계를 모색해야한다. 최근의 정책환경 변화를 고려하여 지방교부세와 국고보조금 제도를 중심으로 한 지방재정지원체계를 개편해야 한다. 사회투자정책의 추진과정에서 관련부문간 재정적 갈등 쟁점이 발생할 수 있다. 예상되는 이해관계 상충부문들에 대한 합리적인 갈등관리 방안들이 사전에 마련되어야 한다.

본문
압축적 저출산?고령화, 연금재정 고갈, 건강보험재정 위기 등과 같이 사회 정책과 관련된 국가재정 지출부문에서는 부정적인 현상과 예측이 매일 같이 발표되고 있다. 한국 사회의 위기에 대한 인식과 국가재정 건전성 유지 가능성에 대한 회의적인 비판들이 많다. 이에 따라 전통적으로 사회적 소비라고 생각 되는 사회복지비에 대해서는 항상 소극적인 수준에서의 국가 예산 배정과 몸집 줄이기, 예산팽창에 대한 비판적인 시각, 그리고 이에 대한 재비판과 설득 등과 같은 일반적인 사회적 논쟁이 이어지고 있다.
전통적인 복지국가의 재정 관점에서 사회복지지출을 접근하면 예상되는 논쟁과 정책의 결과는 항상 동일하다. 재정축소와 이를 반대하는 정치와 운동 속에서 적당한 수준에서 한해 한해의 정부 지출 규모가 결정되는 매우 불안정한 재정관계가 형성?작동되고 있다.
빈곤을 중심으로 하는 전통적인 사회위기에 대한 국가재정 대응 뿐 아니라 소득양극화, 계층 고착화와 대물림, 사회적 배제의 일상화, 양극화와 근로 빈곤층의 양산, 저출산과 고령화, 가족의 해체와 지역사회공동체의 붕괴 등과 같은 새로운 사회위기가 발생하면 국가가 대응해야 하는 사회적 지출은 과거와는 비교할 수 없을 정도로 급증할 것으로 예상된다. 이에 따라 또다른 재정위기에 대한 사회적 우려가 많이 제기되고 있다. 전통적인 복지국가의 재정관리 관점과 수단의 범주에서는 사회부문에서 전통적 위기와 새로운 위기가 복합적으로 제기되고 있는 한국 사회의 재정위기 전망을 극복할 수 있는 대안을 만들지 못한다. 위기의 속도만 일시적으로 늦출 뿐이다.
국가 재정부문에서는 지금 눈 앞에 예상되는 위기를 불가피한 것으로 체념하고 앉아서 지켜볼 수는 없다. 기존의 접근 틀에서는 해결방안이 모색되지 않는다면 새로운 관점을 생각해야 하고 그 타당성을 보다 신중하게 논의? 분석해야 한다. 사회정책부문에서 소극적이고 회피적인 재정대응 보다는 다른 차원에서의 거시적인 인식적·실질적으로 의미있는 대안과 정책이 필요하다. 이러한 시점에서 복지국가 재정위기의 대안으로 사회투자국가의 재정접근이 의미 있게 모색될 필요가 있다. (연구결과나 정책운영의 변화에 따라 약간의 차이는 있을 수 있지만) 연금재정 고갈 예측 시점이나 소득불평등 지수를 생각하면 다행히 아직 한국 사회와 국가 재정에서는 십 수년 이상의 대비 시간이 남아 있다.
사회투자와 관련하여 복잡한 이념과 담론들이 어려운 용어를 동원하여 제시되고 있다. 하지만 국가재정운영에서 사회투자의 기조는 복잡하지 않다. 사회투자정책을 위해 새로운 대안적인 재정관리 관점의 정립이 필요하다는 것은 사회복지분야에서 투자의 대상과 수준 그리고 재정지출의 관리방식에 대한 변화가 필요하다는 것이다. 이는 일반 재정지출 방식과 비교하면 새로운 것이 아니다. 하지만 대상과 수준이 달라진다는 것을 고려하면 기존의 복지 국가 재정과는 근본적인 차이가 있다.
새로운 사회적 혹은 재정적 위기 국면에서 국가 재정이 해결해야하는 두가지 과제는 분명하다. 문제 인식 자체가 대안이 될 수도 있다. 하나는 지속가능한 재정부담(복지재정 총량관리)과 복지지출 확대(개인별 복지서비스 품질 및 규모 확대)를 양립할 수 있는 방안을 모색해야 한다. 이를 위한 사회투자 정책들의 중요성이 인정된다. 사회투자란 기회균등, 예방, 사회적 통합을 지향하는 정책들이고 이들 정책에서는 달성해야하는 목표가 분명히 설정될 수 있기 때문에 보다 효율적인 재정관리가 가능하다.
사회투자정책들은 전통적인 사회적 소비 성격이 강했던 기존 복지에 비해 현명한 복지지출을 요구한다. 이는 개별 복지서비스의 양과 질은 증대되지만 국가의 복지재정 총량은 지속적인 재정부담이 가능한 수준을 유지 할 수 있게 하는 대안이 될 수 있다. 제대로된 성공적인 사회투자정책을 통해 빈곤의 대물림을 억제하고 인적 자산의 사회적?경제적 가치를 높인다면 사회서비스의 실질 대상자의 규모가 자연스럽게 줄어들 수 있다. 이를 통해 사회복지재정의 총량을 부담 가능한 수준에서 관리와 사회서비스 수급자 개인에 대해 양질의 정책들이 양립할 수 있게 된다. 이는 새로운 사회 위기와 이에 대응하는 새로운 사회투자정책에 대한 집중적이고 전략적인 재정지원이 필요한 이유가될 수 있다.
두 번째로 중요한 것으로, 이와 같은 사회투자정책과 복지재정의 지속가능성이 양립할 수 있기 위해서는 재정지출의 수준과 운영 방식에 대한 전환이 필요하다. 즉 전통적인 사회복지지출은 수요관리 차원에서 운영되었다. 국가의 재정형편에 따라 큰 문제가 발생하지 않는 차원에서 적정 수준의 지출에 국한되었다. 이와 같은 지출 접근이 국가 재원 배분과정에 전제되어있었기 때문에 사회복지 재정의 우선순위는 낮을 수밖에 없었다. 지향하는 성과를 창출하기에는 재정규모, 지출대상과 지출방식이 한계가 있다는 사실에 명시적 혹은 암묵적으로 공감을 하기 때문에 특별한 성과관리를 강제하지 못했다. 그저 적당한 수준에서 적당히 사회적 갈등과 문제를 무마하는 것으로 만족해야 했다.
하지만 “투자적 접근”은 이와는 근본적으로 다른 방식이다. “투자”를 위해서는 적정 수준에서 적당히 재원을 배정해서는 안된다. 전략적인 목표를 설정하고 제대된 최적의 지출이 필요하다. 물론 사회투자정책에서 지출의 대상은 사회적 기반 부문이 된다.
<그림 1> 사회재정에서 수요관리와 전략투자영역 구분(예시)
적당한 지출이 아닌 제대로 된 투자는 상대적으로 과거보다는 많은 규모의 재정지출을 수반하게 된다. 이러한 재원은 대부분 납세자들의 세금을 통해 동원된다. 따라서 소비적 지출이 아닌 투자적 지출 부문으로 사회투자가 접근될 경우에는 지출된 재원과 설정된 성과목표에 대한 “납세자의 책임” 의무가 동시에 부과되어야 한다. 적정 수준에서 복지수급권자에 대한 인권차원에서의 온정적 가부장적 혹은 정서적 보호가 아닌 국민의 권리와 국가의 의무로서 제대로 된 투자적 지출이 추진된다면 지향하는 사업의 목표가 보다 구체적으로 제시되고 이들 목표 달성을 위한 전략적 투자 운용이 필요하다. 보다 중요하게는 투자운용 이후의 결과에 대한 “성과책임(납세자 책임)”이 명심적으로 설계되어야 한다. 그러한 성과책임은 일반 시민들이 요구하는 보편적 서비스의 확대와 사회적 기반의 확충 그리고 사회적 및 경제적 안정과 성장잠재력 확충에 대한 약속 이행일 수도 있다.
이와 같은 사회투자에 대한 접근에 대한 사회적 합의가 형성될 경우 국가 재정적으로는 두 가지 큰 변화를 수반하게 된다. 하나는 재원배분에서 국가 우선순위가 바뀌어져야 한다. 누구나 인정하듯이 그리고 자세히 설명할 필요도 없이 우리나라의 재정비중에서 사회복지비는 OECD 최하위 수준이다. 국민연금이 본격적으로 지출되지 않기 때문에 사회복지비 비중을 수평적으로 비교하는 것은 적합하지 않다는 주장도 있지만, (이 문제는 좀더 객관적인 논의가 필요하다) 비전 2030에서 확인할 수 있듯이 사회적 기반에 대한 지출 수준과 사회적 기반의 양과 질이 절대적으로 열악하다는 사실에 대해서는 누구나 공감할 수 있다. 그리고 그동안의 경제성장 정치의 불가피한 부작용이라는 해석에도 공감할 수 있다. 하지만 지식기반경제체제에서 새로운 성장동력으로서 사회적 기반에 대한 중요성을 인식한다면 그리고 한국경제의 경쟁력과 기반은 이미 글로벌 표준 수준에 근접했다는 점에 공감을 한다면 이제 국가의 지출 우선순위는 사회적 기반에 주어져야 한다.
<표 1> OECD 주요 국가의 복지재정 비중
자료 : OECD. (2005). National Account.
두 번째는 정부재정지출의 접근방법에 전제되어 있던 머스그레이브의 재정 원칙에 대한 재정립이 필요하다. 1959년도에 재정학자인 머스그레이브는 신고전경제학에 기초하여 바람직한 정부재정운영의 원칙을 정립하였다. 이는 지난 40여년동안 복지국가 재정원칙의 근간이 되었다. 하지만 복지국가체제의 재정지출 원칙을 정립하였던 당시와 비교하면 재정환경에서 많은 변화가 진행되었다. 이에 따라 사회투자국가의 이행에 맞추어 국가재정원칙에 대한 기존 관점의 재검토가 필요한 시점이 되었다.
<그림 3> 사회투자국가의 재정접근
전통적인 재정원칙은 투자부문(자원배분기능)에서는 파레토 효율과 소비자 주권에 바탕을 둔 최적 지출을 강조한다. 반면 소비부문인 소득분배기능에서는 사회적 가치가 전제로된 적정 수준의 지출이 필요하다. 이러한 전제 속에서 중앙?지방간 재정기능과 우선순위도 설정되었다. 하지만 사회적 기반이 성장의 잠재력을 좌우하는 사회투자국가에서는 전통적인 재정기능에 대한 재검토가 필요하다. 가장 중요하게는 사회적 기반에 지출은 소득분배가 아닌 자원배분기능의 재정 접근을 적용해야 한다는 것이다. 그리고 이를 가능하도록 정부간 재정기능과 재정 우선순위가 재설정되어야 한다는 것이다.
사회복지재정은 『국민의 정부』에서 기초생활보장체계를 정립한 이후부터 의미 있는 수준에서 지속적으로 증대되고 있다. 당시 IMF 경제위기와 사회적 위기를 적극적으로 대응하기 위한 불가피한 선택이었지만 그것만으로 모든 것을 설명하지는 못한다. 빈곤과 사회적 문제에 대한 국민의 권리와 국가의 책임이 필요하다는 인식과 접근의 전환이 이루어진 것이다.
다만 이것은 사회복지정책분야에 국한된 인식이며 재정부문에서는 여전히 시민운동이나 정치권력에 따른 마지못한 지출확대에 그칠 수도 있다. 이와 같은 분위기로 인하여 사회복지재정은 지속적으로 급증하고는 있지만 체계적으로 관리 접근하려는 의미있는 노력은 아직 미흡한 편이다. 사회투자정책이 의미있게 진행되기 위해서는 사회개발 영역에서 국가 재원배분이 보다 강화될 수 있다. 하지만 납세자에 대한 책임까지 고려한 보다 합리적인 사회투자 지출이 가능하기 위해서는 기존의 복지재정 지출구조와 체계 그리고 서비스 전달체계의 효율성과 적합성에 대한 재검토가 필요하다.
90년대 중반이후부터 사회복지재정의 가장 큰 특징은 급속한 규모 증대이다. 1996년도의 4조 5천억원 수준에서 2006년도는 12조원 수준에 육박하였다. 하지만 사회투자의 관점에서는 잊고 있는 다른 더 큰 특징이 있다. 그것은 연도별로 불규칙적이고 사회투자 부문간에도 불균등한 지출구조가 형성되고 있다는 점이다. 또한 사회복지재원의 관리체계가 다원화되고 있지만 이를 통합적으로 운영하려는 관점과 제도는 아직 모색되지 않고 있다.
<그림 4>에서 확인할 수 있듯이 일반회계에서 사회복지비는 IMF 위기 국면에서 몇 년동안 급속히 상승하였다. 2000년대 들어오면서 약간 하락 혹은 주춤하다가 최근 다시 증대되고 있다. 이는 일관된 사회정책 기조가 형성되지 않고 부분적이고 분절적으로 특정 정책 중심으로 복지재정이 운영되기 때문이다. (물론 좀더 세밀한 분석이 필요하다). <표 2>를 보면 10여년 전의 사회복지비는 보건복지부의 재정에 국한되었지만 2006년도에 이르면 보건복지부, 여성가족부, 지방정부(분권교부세), 국민건강기금 등으로 확산되고 보건복지부 내에서도 일반회계 뿐 아니라 다양한 특별회계를 운영하게 된다. 이러한 다변화된 복지재정지출구조가 형성되고 있지만 종합 관리 기능 없이 중앙정부의 일반회계 비중과 GDP 비중은 지속적으로 증대되고 있다.
<그림 4>중앙정부 일반회계 대비 사회복지재정의 비중 추이
자료：표 2와 동일
<표 2> 중앙정부 일반회계 대비 사회복지재정의 비중 추이
주：?사회재정? 항목에는 분권교부세 사업과 여성부의 가족 및 보육사업 예산 포함.
자료：보건복지부. 각년도. ?보건복지부소관 세입?세출 예산 설명자료?
여성부. 2006. ?2006년도 예산개요?; 행정자치부 지방교부세팀 내부자료
통계청. 통계정보시스템 http://kosis.nso.go.kr/
여기서 쟁점으로 지적될 수 있는 것은 불규칙적이고 불안정한 지출구조에 대해 납세자들이 동의하는 사회적 합의가 어떠한 방식으로 이루어지고 있는가 하는 문제이다. 소비적인 수요관리 관점에서는 특별한 사회적 관심과 쟁점을 밝히지 않고 그 해 그 해 임시방편적으로 접근하기 때문에 일반시민들은 감성적 판단의 관점을 가질 수 있다. 이에 따라 균형있는 재정적 시각을 가지는데는 한계가 있다.
또한 재정규모가 급증하지만 국가재정관리의 관점은 과거의 수요관리와 분절적 접근 이상의 의미있는 전략접근은 시도하지 못하고 있다. 대표적으로 국가재정운용계획에 제시된 사회복지분야의 성과목표를 보면, 사회적 의미를 해석하지 못하며 현재 쟁점이 되는 주요 복지시설과 정책에 대해 국가가 나름의 노력은 하고 있다는 성의표시 이상의 접근은 확인되지 않는다. 비전 2030에서 전체적인 사회투자와 사회적 기반에 대한 틀은 제시되었지만 이를 실천하는 재정체계에 대한 신중한 논의들은 아직 활발히 전개되지 못하고 있으며 “돈이 어디 있느냐”라는 퍼주기 선심정책이라는 전통적인 시각에서의 감성적 비판이 많이 제기되고 있다.
<표 3> 국가재정운용계획에서의 사회복지분야 성과관리 목표
주 : 2008년 목표는 2004년 계획서의 목표이며 2009년 목표는 2005년도 계획서에서 제시한 목표치.
자료 : 대한민국정부(2004) ; 대한민국정부(2005)
그동안 각종 경제개발 정책과 선거의 공약을 들여다보면, 재원동원이 쟁점의 정면에 부각되지는 않았다. 이것이 국가 발전을 위한 전략투자 대상이라고 한다면 투자사업의 사회 경제적 의미가 있는지 여부에 초점을 맞추고 이를 효과적으로 달성하기 위해 필요한 세부전략과 행정 및 재정조치에 대한 논의가 뒤를 이었다. (결과적으로) 엉터리 수치와 기대효과가 제시된 경우도 없지는 않았지만 이것은 관려 전략 사업이 달성해야할 목표치라고 양해가 되었다. 기획이 스스로의 재원을 동원하는 형태였다.
하지만 사회복지부문에서는 완전히 반대였다. 관련 정책의 중요성 여부에 대한 관심은 처음부터 설정되지 않는 경향이 있다. 어떻게 하면 희소한 국가 재원을 효율적이고 효과적으로 집행할 수 있는가에 대한 전략적인 고민 보다는 (회의적이고 비판적인 사회재정의 일반 정서 속에서) 한 푼이라도 더 받아내야 한다는 정략적인 노력에 너무나 많은 (정말 더 희소한) 행정 및 재정관리 그리고 사회운동 에너지가 낭비되기도 한다.
유독 사회지출 부문에서는 관련정책이 경제성장에 도움이 되는지를 입증하라는 요구가 강하고 그것은 전략적이고 충분한 (투자) 재원배분을 회피할수 있는 효과적인 수단이 되었다. 전형적인 사회복지정책의 재원배분과정에 대한 시나리오를 생각해보면, 우선 선거를 중심으로한 정치일정이 먼저 고려되고, 돈이 남아도는 것이 아니라는 비판이 앞선다. 뒤이어 시민운동과 시민 정치가 뒤따르고 사회적 아젠더로 성공을 하면 적당히 성의표시하는 수준에서 복지비 지출이 증대된다. “이 정도면 되겠느냐”는 수요관리식 재원배분관행 때문에 일단 돈부터 확보한 이후 거기에 맞추어 나누어 먹기식 재정지출 관행이 형성될 수도 있다. 당연히 전략투자와 성과관리가 요구될 여지는 없으며 누구도 그것이 가능하다고 생각하지 않을 수 있다.
국민소득 2만불 시대의 전통적인 사회 경제적 구조를 뛰어넘는 새로운 도약을 위해 물리적인 경제기반과 사회적 기반의 균형된 형성이 중요하다는 점에 인식을 공유한다면 경제투자와 사회지출 사이에 형성된 기존의 전통적인 재원배분 및 재정관리 체계에 대해서도 재검토와 개편이 필요한 시점이라고 판단된다.
사회투자정책의 재정관리 접근을 모색할 때 가장 중요한 것으로 재원배분의“균형” 관점이 강조되어야 한다. 여기서 균형은 다시 두 가지의 균형으로 구분된다. 하나는 국가재원배분의 거시적 균형이고 다른 하나는 사회지출 부문내의 미시적인 균형이다.
사회투자의 필요성과 중요성에 대한 사회적 합의가 형성된다면, 국가재원배분에서 예산자원의 실링비중에 대해 재설정할 필요가 있다. 이는 국가의 재정기능에 대한 균형 조정을 의미한다. 우리나라는 성장 중심의 재정자원배분이 지속되었기 때문에 국가재정에서 경제개발비의 비중이 항상 높다. 하지만 국가의 경제적 기능이 과거와 같이 노젓기를 계속하는 것이 아니라 시장의 활력과 경쟁력을 극대화하기 위해 신공공관리와 정부혁신에서 강조하듯이 방향잡기로 전환한다면 경제개발에 대해 과거와 같이 많은 재원을 배분할 필요성은 약해진다. 물리적 SOC의 과잉투자와 비효율성 그리고 지역간 나누어 먹기식 병리에 대해 적지 않는 비판이 제기되고 있다. 경제적 한계 생산성이 과거와 같이 절대적 우위를 가지고 있는지 다시 재검토를 한 다음에 재원배분의 적정성을 신중하게 논의할 필요가 있다.
<표 4> 주요 국가의 기능별 재정 지출 비교
자료：기획예산처(2006). 예산개요참고자료 ;(日)自治省(2004), 地方財政自書 ; IMF(2004), Government Finance Statistics Yearbook
<그림 5> 중앙정부 일반회계의 재정기능 추이
자료: 기획예산처. 각연도. 예산개요참고자료
예를 들어 지역사회의 환경변화(인구구조변화 소득수준변화 등)를 고려하면 기존의 사회 경제통계에 기초한 각종 비용편익분석의 결과는 완전히 달라질수 있다. 실현가능성이 희박한 목표치(예, KTX 승객수송률이나 지방공항 이용률)를 미래의 전략 목표치로 변명하기에는 현실과 미래 전망이 너무나 많이 변하고 있지 않는지 좀더 의미있게 고려해야 한다.
<그림 5>에서는 중앙정부 일반회계 세출 기능별 추이가 정리되어 있다. 1991년도부터 지금까지 정부재정기능의 비중 구조를 보면 사회개발비가 비록 의미있는 수준으로 지속 증대되고는 있지만 예산자원배분 구조를 변화시키지는 못하고 있다. 최근 몇 년동안 규모상으로 사회개발비가 증대된 것은 국가재정 규모 증대에 따른 비례적인 결과이며 상대적인 비중에서 의미 있는 구조 변화가 있던 것은 아니다.
국가 예산자원배분 관점에서, 90년대 중반에 방위비와 경제개발비의 우선 순위가 전환되는 1차 구조변화가 발생하였다. 국민의 정부에서 추진했던 대북포용 햇볕정책이 재정구조에 반영된 결과로 해석할 수 있다. 2000년대 들어오면서 사회개발비가 일반행정비 수준을 넘어서는 2차 예산구조 변화가 있었지만 상위수준에서 구조변화가 창출될 것은 아니다. 2000년대 중반에는 경제개발비의 규모와 비중이 축소되는 3차 예산구조 변화가 발생하였는데 이는 사회개발비의 비중 구조 변화가 아닌 교육비 지출 확대에 따른 것이다. 따라서 사회투자의 중요성이 강조되고 있다고는 하지만 국가재원배분(우선 순위정) 구조에서 의미있는 변화는 확인되지 않았다.
사회투자의 재정정책에서는 무엇보다 경제개발과 사회개발 그리고 정규 학교 교육과 사회교육 등의 분야를 중심으로 국가의 예산 자원 배분에서 적정 균형에 대한 체계적 재검토가 필요하다. 사회복지지출 확대에 대한 국가 혹은 사회적 논의에서 항상 먼저 등장하는 것이 “도대체 무슨 돈으로 하려고 하는가? 또는 세금을 올려 새로운 재원을 만들어야 하는가” 라는 논쟁거리가 앞서고 있다. 복지지출을 확대하려면 스스로 재원을 만들어서 추진하라는 소극적이고 비판적인 접근이다. 기존의 재원배분 구조 개편을 통한 국가의 예산자원 배분 우선순위 조정이라는 사회적 필요성 노력 관점은 아직 형성되지 않고 있다. 이는 아직 사회투자 정책이 국가재정 자원배분에서 우선 순위를 가지지 못하고 있다는 것을 의미한다.
거시적 차원 재원배분 우선순위에 대한 균형 조정 과제와 병행하여 사회복지(투자) 부문간의 균형있는 자원배분 노력 역시 매우 중요한 재정관리 과제이다. 경제부문에서 투자와는 달리 사회투자 부문에서는 하위 부문별“균형”적인 서비스 공급과 재원배분이 필요하다는 특징이 있다. 전통적으로 경제분야에서는 경쟁력 기준에서 성장동력(산업)에 대한 “선택과 집중”이 필요하다는 비교우위이론이 설득력을 가진다. 모든 산업부문에 고르게 재원을 배분하는 것은 전형적인 나누어먹기식이 되어 투자의 효과를 기대하기 힘들 수 있기 때문이다.
하지만 사회투자부문은 이와는 다르다. 사회투자의 초점이 가족과 공동체(지역사회) 기반 확충으로 설정될 경우, 관련 하위 부문에서의 균형있는 개발이 필요하다. 대표적인 사례로 사회투자정책으로서 “경쟁력있는 가족 기반” 확충을 들 수 있다. 이 경우 아동?노인?여성 등가족의 구성원에 대한 사회적 기반(사회서비스)가 고르게 갖추어져야 한다. 나누어먹기식 분산 지출이 아닌 상위 목표의 실천 부문에 대한 균형된 정책접근이 필요하다. 이러한 “균형”을 위해서는 다시 두 가지 측면에 유의해야 한다.
<그림 6> 사회투자정책 부문의 복합적 구조
우선 빈곤에 대한 전통적인 사회복지정책 부문과 삶의 기회균등·참여·예방투자를 중심으로 하는 사회투자정책부문간의 균형있는 재원배분 조정이 필요하다. 기초생활보장은 사회투자의 핵심적인 전제조건인 것은 사실이지만 지출의 성격상 사회적 소비 특성이 강하다. 이 부분에 대해서는 수급자가 원하는 만큼의 충분한 재원배분이 이루어지지 않는다. 복지병이라는 비판적 시각에서는 그것은 오히려 바람직하지 않을 수도 있다. 사회적 정의의 가치 전제 속에서 사회와 납세자가 합의하는 적정 수준에서 공공자원이 배분된다. 이 부문에서는 사회적 쟁점이 붉어지는 우연적이고 분절적인 기회가 발생하면 정치와 운동을 통해 추가적인 재원이 확보되기도 한다. 따라서 재정 자원은 사회적 환경과 정치적 관계에 따라 과소와 과다배분이 발생할 가능성이 많은 영역이기도 하다.
반면, 사회투자 영역들은 전략적인 관점에서 최적 투자 지출이 필요하다. 배정된 재원에 대한 납세자 책임은 전략목표와 성과평가를 통해 정당화될 수 있다. 따라서 투자지출의 명분에서 상대적인 우월성이 있어 일관성있는 예산자원 확보에서는 유리할 수 있지만 성과에 대해 재정적인 책임져야 하기 때문에 사업을 수행하거나 쉽지 않은 영역이다. 결국 정치와 운동 그리고 적정한 수준에서의 수요관리로 접근되는 기초적인 사회기반부문과 정책과 성과관리 그리고 전략적이 성과목표와 성과책임 구조가 적용되어야 하는 사회 투자 부문간 재원배분의 균형이 필요하다.
다른 한편으로, 사회투자부문에서의 균형적인 자원배분에서는 사회투자의 하위 부문들간의 균형 있는 재원배분 관점도 중요하다. 사회투자국가에서 인적자산부문은 <그림 7>과 같이 전통적인 정규교육부문 뿐 아니라 출생에서 사망에 이르는 생애 전단계에서 필요한 사회적 기반을 확충하는 것이다. 따라서 생애단계별로 그리고 사회계층(집단)별로 균형 있는 투자자원 배분 관점이 중요하다.
<그림 7> 사회투자정책에서의 인적 자산관리 투자 영역
전통적인 사회복지 재정의 관행에서는 균형의 관점 보다는 사회적 쟁점에 따른 정치적 힘과 운동의 방향에 따라 재정자원의 규모와 속도 그리고 방향이 결정되는 경향이 있다. 이에 따라 정책의 성과와 부문별 기반 형성에서 불균형 쟁점이 발생하게 된다. 예를 들어 최근의 저출산 고령화 쟁점이 부각되면서 노인복지와 보육부문에 집중적인 재원이 배정되지만 여성, 가족, 아동, 장애인에 대한 자원배분은 상대적으로 불리하게 된다. 관행적으로 보면 이들 부문에서는 또다른 독특한 사회적 사건이 발생하게 되면 다시 집중적인 재원배분이 기대될 수 있다. 합리적 재원배분 보다는 정치적이고 분절적인 우연 상황에 따른 재원 배분 관행이 현실 설명력을 가지게 된다. 후자의 접근은 현실적이기는 하지만 규범적으로 바람직하지는 않다. 따라서 이러한 관행 속에서는 한발 앞선 기반적?예방적 투자 보다는 대형사고 하나 발생하면 재원배분이 한 차원 더 높아지는다는 병리적인 접근이 만연하게 된다.
<표 4> 사회재정에서 지출비중이 높은 11개 부문
자료 : 표 2와 동일
<표 5>를 보면, 지난 10여년동안 사회재정에서 재원규모는 급증하였지만 일부 현금 급여(지원) 서비스에 집중되어 있다는 것을 확인할 수 있다. 건강 보험, 의료보호, 그리고 기초보호가 76.7%를 차지하며 상위 11개 프로그램이 94% 정도를 차지할 정도로 재정자원배분은 불균형적이다. 여기에 2008년도부터 시행되는 기초노령연금까지 포함되면 현금급여 부문의 비중은 더욱 증대될 것으로 예상된다. 인적자산관리의 취지에 부합하는 사회적 서비스(투자) 지출의 재원 비중은 아직 의미 있는 수준에서 자리매김되지 못하고 있다.
예산재원 확보가 사회투자부문의 궁극 목표는 분명 아니다. 새로운 사회 위기에 따른 새로운 투자확대는 기존의 사회복지부문에 배분된 재원 총량 규모내에서 조정되는 것이 아니다. 기존의 사회복지비 지출은 대부분 법정 의무 부담 형태가 있기 때문이다. 새로운 투자지출에서는 새로운 추가적인 재원배분이 불가피하다. 하지만 지출과 재정관리의 접근방식은 달라진다.
전략투자에 대한 전략적 재원배분에서는 체계적인 성과관리 관점이 구축되어야 한다. 이를 위해서는 다시 두 가지 실천 수단이 모색되어야 한다. 하나는 전략목표의 수준 설정이다. 예를 들어 저소득층 아동의 방과후 학습 프로그램 지원이 사회투자를 위해 중요한 것은 사실이다. 하지만 전략목표를 저소득층 아동의 사회적 일탈 예방으로 설정할 것인지 아니면 중산층 아동 수준의 학력 발달로 할 것인지에 따라 국가가 부담해야할 재원 수준의 규모가 달라진다. “제대로”지출이 필요하다고 하지만 “제대로”의 수준에 대한 사회적합의 형성이 쉽지 않다. 적당한 수준에서의 의사결정이 되면 결국 투자재원의 비효율적 지출은 불가피할 수 있다.
다른 하나는 성과관리의 초점을 설계하는 것이다. 성과관리에서는 투입이 아니라 결과지향적 성과관리가 중요하다는 것은 이미 지난 몇 년간의 정부 혁신과 성과관리 노력에서 합의되었으며 또한 실천을 위해 많은 노력이 있다. 2007년도부터 적용된 국가재정법에서도 제1조에 성과 지향성이 선언되어 있다. 하지만 이 외에도 보다 중요하게, 사회투자에서는 (재원배분에서와 마찬가지로)<표 6>과 같은 프로그램 구조 관점에서의 균형된 성과관리 접근이 필요하다. 특정 사업 중심으로 성과를 관리하면 균형있는 사회기반 형성에 한계가 있다. 즉 사업별로는 문제가 없지만 해당 사업이 궁극적으로 지향해야하는 상위목표달성과는 거리가 있을 수 있기 때문이다.
<표 6> 성과관리를 위한 프로그램구조 (안)
(A) 프로그램 구조
(B) 프로그램별 전략목표와 관리방식
신기하고 대표적인 뭔가 괜찮은 인기사업 중심으로 집중적인 투자를 하면 정치적 혹은 상징적 의미는 충분히 인정될 수 있으나 프로그램 전체의 관점에서는 한계가 있다. 예를 들어 <표 6>에서 노인복지를 위해 장기요양 시설이 중요하지만 궁극적으로 지향하는 “노인건강관리”를 해결하지는 못한다. 해당 사업의 주요 시책들이 전체적으로 균형을 이루어야 지향하는 사회적 성과관리가 가능하다.
사회투자정책은 대부분 사람에 대한 투자이며, 사람은 지역에 기반하여 경제적 사회적으로 활동한다. 따라서 이들에 대한 사회적 기반확충과 지원은“사람”이 살고 있는 현장 중심의 정책이 매우 중요하다. 따라서 경제정책과는 달리 사회정책에서는 지방정부의 역할이 보다 중요하다. 여기까지는 공통적으로 합의가 가능하다. 하지만 지방정부의 어떠한 역할인가에 대해 좀더 구체적으로 내려가면 합의가 쉽지 않다. 행정과 집행활동에서의 노력과 재정부담의 문제는 다르기 때문이다.
사람의 거주 및 경제적 활동은 특정 장소 혹은 공간에 귀속되지 않기 때문에 관련 투자에서는 재정적 외부성 쟁점이 발생한다. 따라서 외부성을 내부화하기 위한 중앙정부의 이전재정은 사회투자의 성공을 위해 매우 중요하다. 예를 들어 사회투자에서 아동교육에 대한 지출이 중요하다. 지방정부가 지역아동의 방과후 학습에 집중적인 투자를 하였지만 그 아동이 성장 후 다른 지역에 거주하게 되면 과거의 지방정부 지출은 재정누출이 된다. 장래 다른 지역에 거주할 아동에게 현재 거주하는 주민들의 세금으로 서비스를 제공하는 것은 합리적이지 못할 수 있다는 외부효과에 대한 경제적 논리가 있다. 이러한 구조 속에서 지방정부는 당연히 자체 재원 중심의 적극적인 재정지출을 하지 못한다. 이와 같은 이유로 사회복지분야에서는 국고보조금의 비중이 높아야 한다는 점에서는 학자들간에 합의가 형성되어 있다.
그런데, 우리나라의 중앙?지방정부간 사회재정관계를 살펴보면 사회투자의 관점 보다는 지방자치 이전 중앙정부의 일선행정기관에 대한 관점이 여전히 지배적이고 그 당시 적용되었던 제도들이 행정편의적인 관행 속에서 지속되는 경향이 있다. 지방정부의 복지재정 부담이 많지 않았던 시절에는 다소의 이의제기와 약간의 불만족 속에서 어느 정도까지는 용인이 되었지만 최근 몇 년동안 사회복지비가 급증하면서 제대로된 정부간 사회(복지)재정관계로 개편되어야 한다는 필요성이 부각되고 있다. 정부간 이전재정관계에서는 무엇보다 지방교부세와 국고보조금체계에 대한 개편이 필요하다.
<표 7> 일반회계 세입 추이
주 : 2004년까지는 결산액, 2005년도 최종예산액, 2006년도 당초예산액
자료 : 자치종합정보센터 (www.jachi.co.kr)
<그림 8> 지방자치단체 일반회계 세출기능별 추이
우선, 정부부처별 국고보조금과 지방비 배분 규모를 살펴보면 2006년 당초예산기준으로 중앙정부 보조금의 31.3%를 차지하는 보건복지부의 국고보조사업에서 지방비 부담비율은 26.1%이다. 지방정부가 상대적으로 선호한다는 건설교통부의 경우는 절반 수준인 14.0%이다. (물론 좀더 자세한 분석이 필요하지만) 외부성에 대한 경제학의 관점에서는 정부간 재정부담 구조가 반대로 형성되어 있는 것이다.
<표 8> 중앙정부 주요 부처별 국고보조금 현황(2006)
주 : 2006년도 일반회계 당초예산액
자료 : 자치종합정보센터 (www.jachi.co.kr)
둘째, 지방교부세의 기준재정수요에서 사회복지비의 비중은 20% 내외로 일반행정관리비의 40%의 절반 수준에 불과하다. 사회복지에 대한 지방비 부담규모가 급증하고 있지만 중앙정부가 인정하는 지방의 기준재정수요에서 사회복지비는 아직 부차적인 중요성만 인정될 뿐이다. 기준재정수요의 특정 단위에 대한 부분적인 개편 작업은 매년 추진되고 있지만 사회투자의 관점에서 재원배분의 결과에서 의미는 한계가 있다.
<표 9> 재정특성별?정책분야별 기준보조율 설계구조(안)
셋째, 20여년전에 만들어진 “보조금의 관리 및 예산에 관한 법률”에서의 기준보조율이 지방자치 실시와 사회 경제적 환경변화가 많았던 지금의 현실에도적실성을 가지는지 여부에 대한 논의가 필요하다. 현행 기준보조율을 살펴보면, 농업부문에서는 국고보조율이 100%인 사업이 다수가 있고 사회복지 부분에서는 서울특별시만 적용되는 50% 보조금 사업이 많다. 지방정부의 재량적인 선택의 여지가 없는 법정 의무지출(부담) 보조금에 대해서도 지방 정부가 지방비를 부담해야할 근거를 찾기가 쉽지 않다.
예를 들어 국민기초생활보장체제의 확립과 함께 저소득층에 대한 기초생활보장은 온정적 시혜에서 국민의 권리로 전환되었다. 하지만 예산단위 사업에서는 생활보호가 생계급여로 바뀌고 의료보호가 의료급여로 그리고 경로수당이 기초노령연금으려 명칭이 개편된 것 이외 정부간 재정관계 측면에서 의미있는 구조적 변화는 없다. 시혜에서 권리로 개편되고 지방정부가 재량적으로 선택?조정할 여지가 없는 지출분야에서는 재정부담 주체인 중앙정부가 관련 예산의 100%를 부담할 필요가 있다. 이 분야에서 현재의 지방비 부담을 강제하는 합리적인 근거를 찾기가 쉽지 않다.
해당 사업이 중앙과 지방의 공통 사무라고 전제하여 지방비 부담이 필요하다는 주장이 있다. 하지만 공통사무라는 해석은 중앙정부의 시각에서 설정한 것이며 지방정부는 관련 입법 과정에 참여하여 의사를 반영할 여지는 매우 협소하다. 지방비 부담에 대한 또다른 이유로, 사회복지서비스의 경우, 100% 중앙정부 지원으로 사업을 운영하면 부적격 수급자에 대한 통제적인 재정 관리 노력을 소홀히 할 우려가 있다는 것이다. 현실 관행을 고려하면 설득력이 없는 것은 아니지만, 전달체계에서 오차율관리는 재원부담 비율 구조가 아닌 별도의 행정관리수단으로 접근해야할 사안이다.
넷째, 신청주의와 지방비 매칭의 획일적 적용 원칙 때문에 국가적인 표준 수준에서 공급되어야 하는 사회복지서비스의 지역간 불균등 문제 역시 심각하다. 지자체의 재정수준에 상관없이 사회적 정의 차원에서 동등한 기초적인 사회 기반 구축이 필요하지만 현재의 정부간 재정구조에서는 이를 실천하기는 쉽지 않다.
사회투자를 위한 국가재정관리체계와 재원배분 구조가 개편된다면 정부간 사회재정관계의 재정비 작업도 동시에 추진되면서 서로간의 적정 균형을 찾아야 한다. 지방정부는 원래 복지지출을 싫어하는 비윤리적이고 비도덕적이고 비복지적인 공공부문이라는 비판이 과연 현실적으로 정확한 것인지 다시 생각해 봐야 한다. 사회개발비가 경제개발비를 훨씬 넘어서고 있는 최근의 상황에서 지방정부가 복지비 부담을 회피한다는 비판은 정부간 재정 관계의 현실에 대한 정확한 인식이 될 수 없다. 민선단체장 체제이후 지방자치가 실시된 지도 10여년이 지나고 있지만 정부간 재정관계에서는 새로운 환경변화를 제대로 반영하지 못하고 있다.
사회투자정책에 대한 재정우선순위 전환 과제는 불가피하게 기존의 재정 관리 구조와 재원배분 구조에 대한 균형 변화를 창출한다. 재정은 곧 현실인만큼 이에 따른 사회적 갈등 역시 무시하기 힘들다. 따라서 예상가능한 재정 갈등에 대해 사전에 확인하고 합리적인 극복방안들을 모색하는 작업 역시 재정구조 개편정책에서 매우 중요한 과제이다.
첫째, 무엇보다 복지재정지출 확대와 국가재정 건전성 쟁점이 있다. 이는 사회투자 정책을 퍼주기 선심성 예산으로 볼 것인가 전략적인 사회적 기반 확충 예산으로 볼 것인가의 차이에서 갈등해결 방안을 모색해야 한다. 이를 위해 미래에 다가올 재정재앙에 대해 회피적이고 소극적인 예산인지 보다는 적극적이고 성과에 대해 납세자에게 책임지는 “사회투자 인지예산” 관점이 무엇보다 중요하다.
둘째, 전통적인 사회복지재정 접근과 사회투자정책의 상충성 쟁점이 예상된다. 하지만 효과적인 사회투자정책의 성과관리를 위해서는 기초적인 사회복지기반이 전제가 되어야 하기 때문에 이 두 부분은 상충관계에 있다기 보다는 보완관계에 있는 것이다. 정치적 상징성과 매력이 있다고 하여 사회 투자 부문에 과도한 재원 배분이 집중되거나 사회운동과 체감 혹은 온정적 시혜로 인해 기초복지에 대한 소비적 지출이 과도해서는 곤란하다. 무엇보다 사회재정 부문별 균형있는 재원 배분 관점이 중요하다.
셋째, 결과지향적인 사회투자정책의 성과관리는 담당조직의 긴장을 유발하게 된다. 결과지향적인 정책은 지식기반행정이 중요하다. 하지만 이는 물리적 투자와는 달리 단기간에 구축되지는 못한다. “사람”에 대한 “사람”의 정책과제라는 점이 현실적으로 인정되어야 한다. 따라서 점진적이고 지속적 변화, 관리와 통제 보다는 지원과 역량강화 노력 접근이 중요하다. 정책에 대한 일관성 유지와 사회적 인내가 무엇보다 요구되는 부분이기도 하다. 미국의 통합성과관리체계에 대한 개혁입법인 “정부성과 및 결과에 관한 법”(GPRA: Government Performance and Result Act of 1993)은 7년간의 시범사업기간을 걸쳐 완성되었다. 마음만 앞선 정책은 부작용만 더 키울 수 있다는 점이 충분히 고려되어야 한다.'''

In [67]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)

In [68]:
pages = text_splitter.split_text(contexts)
pages[1]

'본문\n압축적 저출산?고령화, 연금재정 고갈, 건강보험재정 위기 등과 같이 사회 정책과 관련된 국가재정 지출부문에서는 부정적인 현상과 예측이 매일 같이 발표되고 있다. 한국 사회의 위기에 대한 인식과 국가재정 건전성 유지 가능성에 대한 회의적인 비판들이 많다. 이에 따라 전통적으로 사회적 소비라고 생각 되는 사회복지비에 대해서는 항상 소극적인 수준에서의 국가 예산 배정과 몸집 줄이기, 예산팽창에 대한 비판적인 시각, 그리고 이에 대한 재비판과 설득 등과 같은 일반적인 사회적 논쟁이 이어지고 있다.\n전통적인 복지국가의 재정 관점에서 사회복지지출을 접근하면 예상되는 논쟁과 정책의 결과는 항상 동일하다. 재정축소와 이를 반대하는 정치와 운동 속에서 적당한 수준에서 한해 한해의 정부 지출 규모가 결정되는 매우 불안정한 재정관계가 형성?작동되고 있다.\n빈곤을 중심으로 하는 전통적인 사회위기에 대한 국가재정 대응 뿐 아니라 소득양극화, 계층 고착화와 대물림, 사회적 배제의 일상화, 양극화와 근로 빈곤층의 양산, 저출산과 고령화, 가족의 해체와 지역사회공동체의 붕괴 등과 같은 새로운 사회위기가 발생하면 국가가 대응해야 하는 사회적 지출은 과거와는 비교할 수 없을 정도로 급증할 것으로 예상된다. 이에 따라 또다른 재정위기에 대한 사회적 우려가 많이 제기되고 있다. 전통적인 복지국가의 재정관리 관점과 수단의 범주에서는 사회부문에서 전통적 위기와 새로운 위기가 복합적으로 제기되고 있는 한국 사회의 재정위기 전망을 극복할 수 있는 대안을 만들지 못한다. 위기의 속도만 일시적으로 늦출 뿐이다.\n국가 재정부문에서는 지금 눈 앞에 예상되는 위기를 불가피한 것으로 체념하고 앉아서 지켜볼 수는 없다. 기존의 접근 틀에서는 해결방안이 모색되지 않는다면 새로운 관점을 생각해야 하고 그 타당성을 보다 신중하게 논의? 분석해야 한다. 사회정책부문에서 소극적이고 회피적인 재정대응 보다는 다른 차원에서의 거시적인 인식적·실질적으로 의미있는 대안과 정책이 필요하다. 이러한 시점에서 복지국가 재

In [69]:
system_prompt = '''주어진 문서를 이용하여 답변할 수 있는 간단한 질문 5개를 파이썬 문자열 리스트 형태로 제안하세요. 리스트 형태로 바로 작성을 시작하면 됩니다.'''

user_prompt = []
for doc in pages:
  user_prompt.append('문서: ' + doc + '\n질문 5개를 만드세요.\n시작!\n질문 리스트:')

In [70]:
print(user_prompt[1])

문서: 본문
압축적 저출산?고령화, 연금재정 고갈, 건강보험재정 위기 등과 같이 사회 정책과 관련된 국가재정 지출부문에서는 부정적인 현상과 예측이 매일 같이 발표되고 있다. 한국 사회의 위기에 대한 인식과 국가재정 건전성 유지 가능성에 대한 회의적인 비판들이 많다. 이에 따라 전통적으로 사회적 소비라고 생각 되는 사회복지비에 대해서는 항상 소극적인 수준에서의 국가 예산 배정과 몸집 줄이기, 예산팽창에 대한 비판적인 시각, 그리고 이에 대한 재비판과 설득 등과 같은 일반적인 사회적 논쟁이 이어지고 있다.
전통적인 복지국가의 재정 관점에서 사회복지지출을 접근하면 예상되는 논쟁과 정책의 결과는 항상 동일하다. 재정축소와 이를 반대하는 정치와 운동 속에서 적당한 수준에서 한해 한해의 정부 지출 규모가 결정되는 매우 불안정한 재정관계가 형성?작동되고 있다.
빈곤을 중심으로 하는 전통적인 사회위기에 대한 국가재정 대응 뿐 아니라 소득양극화, 계층 고착화와 대물림, 사회적 배제의 일상화, 양극화와 근로 빈곤층의 양산, 저출산과 고령화, 가족의 해체와 지역사회공동체의 붕괴 등과 같은 새로운 사회위기가 발생하면 국가가 대응해야 하는 사회적 지출은 과거와는 비교할 수 없을 정도로 급증할 것으로 예상된다. 이에 따라 또다른 재정위기에 대한 사회적 우려가 많이 제기되고 있다. 전통적인 복지국가의 재정관리 관점과 수단의 범주에서는 사회부문에서 전통적 위기와 새로운 위기가 복합적으로 제기되고 있는 한국 사회의 재정위기 전망을 극복할 수 있는 대안을 만들지 못한다. 위기의 속도만 일시적으로 늦출 뿐이다.
국가 재정부문에서는 지금 눈 앞에 예상되는 위기를 불가피한 것으로 체념하고 앉아서 지켜볼 수는 없다. 기존의 접근 틀에서는 해결방안이 모색되지 않는다면 새로운 관점을 생각해야 하고 그 타당성을 보다 신중하게 논의? 분석해야 한다. 사회정책부문에서 소극적이고 회피적인 재정대응 보다는 다른 차원에서의 거시적인 인식적·실질적으로 의미있는 대안과 정책이 필요하다. 이러한 시점에서 복지국가 재정

In [71]:
# 질문 내용을 토대로 질문지 만드는 건 gpt한테 부탁
response = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt[0]}
  ],
  temperature=0
)

question_lst = response.choices[0].message.content
question_lst

'[\n    "사회투자정책이 중요한 이유는 무엇인가요?",\n    "사회투자정책에서 \'성과에 책임지는 최적의 투자\'란 무엇을 의미하나요?",\n    "사회개발과 경제개발의 균형있는 예산자원배분이 왜 필요한가요?",\n    "중앙과 지방정부 간의 합리적 재정관계는 어떻게 모색할 수 있나요?",\n    "지방재정지원체계 개편의 필요성은 무엇인가요?"\n]'

In [72]:
print(question_lst)

[
    "사회투자정책이 중요한 이유는 무엇인가요?",
    "사회투자정책에서 '성과에 책임지는 최적의 투자'란 무엇을 의미하나요?",
    "사회개발과 경제개발의 균형있는 예산자원배분이 왜 필요한가요?",
    "중앙과 지방정부 간의 합리적 재정관계는 어떻게 모색할 수 있나요?",
    "지방재정지원체계 개편의 필요성은 무엇인가요?"
]


In [73]:
# 파이썬 리스트 모양을 하고 있는 문자열을 실제 파이썬 리스트로 만들기
question_lst = ast.literal_eval(question_lst)
question_lst

['사회투자정책이 중요한 이유는 무엇인가요?',
 "사회투자정책에서 '성과에 책임지는 최적의 투자'란 무엇을 의미하나요?",
 '사회개발과 경제개발의 균형있는 예산자원배분이 왜 필요한가요?',
 '중앙과 지방정부 간의 합리적 재정관계는 어떻게 모색할 수 있나요?',
 '지방재정지원체계 개편의 필요성은 무엇인가요?']

# 데이터 만들기

In [ ]:
# 청킹된 데이터를 데이터 프레임으로 만들기
data = pages

df = pd.DataFrame(data, columns=['text'])
df

,text
0,사회투자정책과 재정관리\n\n\n이재원;\n초록\n\n지식기반 경제체제로의 전환과 ...
1,"본문\n압축적 저출산?고령화, 연금재정 고갈, 건강보험재정 위기 등과 같이 사회 정..."
2,사회투자정책들은 전통적인 사회적 소비 성격이 강했던 기존 복지에 비해 현명한 복지지...
3,<표 1> OECD 주요 국가의 복지재정 비중\n자료 : OECD. (2005). ...
4,<그림 4>중앙정부 일반회계 대비 사회복지재정의 비중 추이\n자료：표 2와 동일\n...
5,국민소득 2만불 시대의 전통적인 사회 경제적 구조를 뛰어넘는 새로운 도약을 위해 물...
6,사회투자의 재정정책에서는 무엇보다 경제개발과 사회개발 그리고 정규 학교 교육과 사회...
7,<그림 7> 사회투자정책에서의 인적 자산관리 투자 영역\n전통적인 사회복지 재정의 ...
8,<표 6> 성과관리를 위한 프로그램구조 (안)\n(A) 프로그램 구조\n(B) 프로...
9,"<표 9> 재정특성별?정책분야별 기준보조율 설계구조(안)\n셋째, 20여년전에 만들..."


In [ ]:
df['document_embedding'] = df.progress_apply(lambda row: sentence_vectorizer.encode(row.text), axis = 1)
df

100%|██████████| 11/11 [00:00<00:00, 14.99it/s]


,text,document_embedding
0,사회투자정책과 재정관리\n\n\n이재원;\n초록\n\n지식기반 경제체제로의 전환과 ...,"[-0.036768895, -0.005391997, 0.005716899, -0.0..."
1,"본문\n압축적 저출산?고령화, 연금재정 고갈, 건강보험재정 위기 등과 같이 사회 정...","[-0.028910842, 0.0081093, -0.03951096, -0.0404..."
2,사회투자정책들은 전통적인 사회적 소비 성격이 강했던 기존 복지에 비해 현명한 복지지...,"[-0.030956907, 0.009449689, -0.03856668, -0.02..."
3,<표 1> OECD 주요 국가의 복지재정 비중\n자료 : OECD. (2005). ...,"[-0.020935865, 0.011895553, -0.04910796, -0.02..."
4,<그림 4>중앙정부 일반회계 대비 사회복지재정의 비중 추이\n자료：표 2와 동일\n...,"[-0.033851486, 0.003232164, -0.040852994, -0.0..."
5,국민소득 2만불 시대의 전통적인 사회 경제적 구조를 뛰어넘는 새로운 도약을 위해 물...,"[-0.039413836, 0.02126285, -0.025439158, -0.01..."
6,사회투자의 재정정책에서는 무엇보다 경제개발과 사회개발 그리고 정규 학교 교육과 사회...,"[-0.0124407, 0.007030219, -0.019498866, -0.015..."
7,<그림 7> 사회투자정책에서의 인적 자산관리 투자 영역\n전통적인 사회복지 재정의 ...,"[-0.022361778, 0.0045888685, -0.03959542, -0.0..."
8,<표 6> 성과관리를 위한 프로그램구조 (안)\n(A) 프로그램 구조\n(B) 프로...,"[-0.040464617, 0.012873567, -0.02628473, -0.02..."
9,"<표 9> 재정특성별?정책분야별 기준보조율 설계구조(안)\n셋째, 20여년전에 만들...","[-0.035722356, 0.030903032, -0.003358286, -0.0..."


In [ ]:
# 어떤 질문을 해야 할지 사람이 아는 경우도 있지만, 모르는 경우도 있음. 그래서 질문을 gpt로 생성함
print(question_lst[0])

사회투자정책이 중요한 이유는 무엇인가요?


In [ ]:
sample_df = return_answer_candidate(df, question_lst[0])
sample_df

,text,document_embedding,similarity
2,사회투자정책들은 전통적인 사회적 소비 성격이 강했던 기존 복지에 비해 현명한 복지지...,"[-0.030956907, 0.009449689, -0.03856668, -0.02...",0.616729
8,<표 6> 성과관리를 위한 프로그램구조 (안)\n(A) 프로그램 구조\n(B) 프로...,"[-0.040464617, 0.012873567, -0.02628473, -0.02...",0.599641
6,사회투자의 재정정책에서는 무엇보다 경제개발과 사회개발 그리고 정규 학교 교육과 사회...,"[-0.0124407, 0.007030219, -0.019498866, -0.015...",0.597153
0,사회투자정책과 재정관리\n\n\n이재원;\n초록\n\n지식기반 경제체제로의 전환과 ...,"[-0.036768895, -0.005391997, 0.005716899, -0.0...",0.583823
1,"본문\n압축적 저출산?고령화, 연금재정 고갈, 건강보험재정 위기 등과 같이 사회 정...","[-0.028910842, 0.0081093, -0.03951096, -0.0404...",0.569052


In [ ]:
system_prompt = """당신은 검색된 문서부터 질문의 답변을 작성하는 언어 모델입니다.

### 지시사항
1. 질문으로부터 주어진 다수의 문서에서 답을 찾아 작성하세요.
2. 검색된 문서에 질문의 답이 없는 경우에는 질문에 대한 내용을 언급하면서 답을 찾을 수 없다고 작성하세요.
3. 답변에서 참고자료의 내용을 인용한 경우, 답변 맨 끝에 인용한 Source_id를 추가하세요. 답변에 내용을 인용한 경우에만 작성해야 합니다. 인용한 게 없다면 쓰지마세요.
4. Source_id 값은 검색 결과에서 가져온 것이며 두 개의 대괄호로 묶어야 합니다. 예: [[d97b811489b73f46c8d2cb1bc888dbbe]], [[b6be48868de736b90363d001c092c019]]
5. 인용한 경우 Source_id를 덧붙이는 것은 매우 중요합니다. 이는 반드시 지켜져야 합니다."""

In [ ]:
search_lst = []

for i, c in zip(sample_df.index.to_list(), sample_df['text'].to_list()):
  search_lst.append(["Source_id" + ": " + str(i), "Content" + ": " + str(c)])

In [ ]:
search_result = ''
for s in search_lst:
  search_result += str(s) + '\n'

print(search_result)

['Source_id: 2', 'Content: 사회투자정책들은 전통적인 사회적 소비 성격이 강했던 기존 복지에 비해 현명한 복지지출을 요구한다. 이는 개별 복지서비스의 양과 질은 증대되지만 국가의 복지재정 총량은 지속적인 재정부담이 가능한 수준을 유지 할 수 있게 하는 대안이 될 수 있다. 제대로된 성공적인 사회투자정책을 통해 빈곤의 대물림을 억제하고 인적 자산의 사회적?경제적 가치를 높인다면 사회서비스의 실질 대상자의 규모가 자연스럽게 줄어들 수 있다. 이를 통해 사회복지재정의 총량을 부담 가능한 수준에서 관리와 사회서비스 수급자 개인에 대해 양질의 정책들이 양립할 수 있게 된다. 이는 새로운 사회 위기와 이에 대응하는 새로운 사회투자정책에 대한 집중적이고 전략적인 재정지원이 필요한 이유가될 수 있다.\n두 번째로 중요한 것으로, 이와 같은 사회투자정책과 복지재정의 지속가능성이 양립할 수 있기 위해서는 재정지출의 수준과 운영 방식에 대한 전환이 필요하다. 즉 전통적인 사회복지지출은 수요관리 차원에서 운영되었다. 국가의 재정형편에 따라 큰 문제가 발생하지 않는 차원에서 적정 수준의 지출에 국한되었다. 이와 같은 지출 접근이 국가 재원 배분과정에 전제되어있었기 때문에 사회복지 재정의 우선순위는 낮을 수밖에 없었다. 지향하는 성과를 창출하기에는 재정규모, 지출대상과 지출방식이 한계가 있다는 사실에 명시적 혹은 암묵적으로 공감을 하기 때문에 특별한 성과관리를 강제하지 못했다. 그저 적당한 수준에서 적당히 사회적 갈등과 문제를 무마하는 것으로 만족해야 했다.\n하지만 “투자적 접근”은 이와는 근본적으로 다른 방식이다. “투자”를 위해서는 적정 수준에서 적당히 재원을 배정해서는 안된다. 전략적인 목표를 설정하고 제대된 최적의 지출이 필요하다. 물론 사회투자정책에서 지출의 대상은 사회적 기반 부문이 된다.\n<그림 1> 사회재정에서 수요관리와 전략투자영역 구분(예시)\n적당한 지출이 아닌 제대로 된 투자는 상대적으로 과거보다는 많은 규모의 재정지출을 수반하게 된다. 이

In [ ]:
user_prompt = '''검색 결과:\n''' + search_result + '질문: ' + question_lst[0] + '\n답변:'''
print(user_prompt)

검색 결과:
['Source_id: 2', 'Content: 사회투자정책들은 전통적인 사회적 소비 성격이 강했던 기존 복지에 비해 현명한 복지지출을 요구한다. 이는 개별 복지서비스의 양과 질은 증대되지만 국가의 복지재정 총량은 지속적인 재정부담이 가능한 수준을 유지 할 수 있게 하는 대안이 될 수 있다. 제대로된 성공적인 사회투자정책을 통해 빈곤의 대물림을 억제하고 인적 자산의 사회적?경제적 가치를 높인다면 사회서비스의 실질 대상자의 규모가 자연스럽게 줄어들 수 있다. 이를 통해 사회복지재정의 총량을 부담 가능한 수준에서 관리와 사회서비스 수급자 개인에 대해 양질의 정책들이 양립할 수 있게 된다. 이는 새로운 사회 위기와 이에 대응하는 새로운 사회투자정책에 대한 집중적이고 전략적인 재정지원이 필요한 이유가될 수 있다.\n두 번째로 중요한 것으로, 이와 같은 사회투자정책과 복지재정의 지속가능성이 양립할 수 있기 위해서는 재정지출의 수준과 운영 방식에 대한 전환이 필요하다. 즉 전통적인 사회복지지출은 수요관리 차원에서 운영되었다. 국가의 재정형편에 따라 큰 문제가 발생하지 않는 차원에서 적정 수준의 지출에 국한되었다. 이와 같은 지출 접근이 국가 재원 배분과정에 전제되어있었기 때문에 사회복지 재정의 우선순위는 낮을 수밖에 없었다. 지향하는 성과를 창출하기에는 재정규모, 지출대상과 지출방식이 한계가 있다는 사실에 명시적 혹은 암묵적으로 공감을 하기 때문에 특별한 성과관리를 강제하지 못했다. 그저 적당한 수준에서 적당히 사회적 갈등과 문제를 무마하는 것으로 만족해야 했다.\n하지만 “투자적 접근”은 이와는 근본적으로 다른 방식이다. “투자”를 위해서는 적정 수준에서 적당히 재원을 배정해서는 안된다. 전략적인 목표를 설정하고 제대된 최적의 지출이 필요하다. 물론 사회투자정책에서 지출의 대상은 사회적 기반 부문이 된다.\n<그림 1> 사회재정에서 수요관리와 전략투자영역 구분(예시)\n적당한 지출이 아닌 제대로 된 투자는 상대적으로 과거보다는 많은 규모의 재정지출을 수반하

In [ ]:
response = client.chat.completions.create(
  model="gpt-4o",
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
  ]
)

print(response.choices[0].message.content)

사회투자정책이 중요한 이유는 여러 가지 긴급한 사회적, 경제적 문제들에 대한 해결책을 제공하기 때문입니다. 이는 전통적인 복지정책과 달리 미래의 성장 잠재력을 확충하고 보다 지속 가능한 사회를 구축하기 위해 설계되었습니다.

1. **지속 가능한 성장**: 사회투자정책은 지식기반 경제체제로의 전환과 맞물려 지속 가능한 국가 성장동력을 확보하기 위해 중요합니다. 기존의 복지지출과 달리 성과에 책임지는 최적의 투자를 통해 사회적 기반을 확충하는 것이 목표입니다[[b6be48868de736b90363d001c092c019]].

2. **빈곤 억제 및 인적 자산 가치 증진**: 성공적인 사회투자정책은 빈곤의 대물림을 억제하고 교육, 건강 등을 통해 인적 자산의 사회적, 경제적 가치를 높이는데 기여할 수 있습니다. 이는 결과적으로 사회서비스 수급대상자의 규모를 줄여서 사회복지재정의 지속성을 향상시킵니다[[2]].

3. **효율적 재정관리**: 사회투자정책은 기회균등, 예방, 사회적 통합을 목표로 하는 정책으로, 명확한 목표 설정이 가능하여 보다 효율적인 재정관리가 가능합니다. 이는 새로운 사회적 위기와 재정적 위기에 효과적으로 대응하는 데 필요한 기반을 제공합니다[[1]].

4. **사회적 기반의 확충**: 사회적 기반의 확충은 경제적 및 사회적 안정과 성장의 기초가 되며, 국가의 경쟁력을 글로벌 표준 수준으로 끌어올리는 데 기여할 수 있습니다[[d97b811489b73f46c8d2cb1bc888dbbe]].

이러한 이유로, 사회투자정책은 단기적인 소비성 복지 지출에 그치지 않고 장기적인 사회적, 경제적 이익을 제공하는 방향으로 나아가야 합니다.


# Cohere CoT


In [ ]:
system_prompt = """

당신은 검색된 문서부터 질문의 답변을 작성하는 언어 모델입니다.

### 지시사항
1. 질문으로부터 주어진 다수의 문서인 검색 결과에서 답을 찾아 작성하세요.
2. 검색된 문서에 질문의 답이 없는 경우에는 질문에 대한 내용을 언급하면서 답을 찾을 수 없다고 작성하세요.
3. 검색 결과의 문서에서 답변을 인용한 경우 인용한 부분을 <co: Source_id> 와 </co: Source_id> 기호를 사용해서 감싸서 표시하세요. 감싸지 않고서는 정보를 전달하려 하지 마십시오.
4. 예를 들어 답변을 Source_id : 0 에서 인용했다면 <co: 0>답변</co: 0> 처럼 표기합니다.
5. 그 어떠한 정보도 인용없이 설명하려 하지마십시오. <co: Source_id> </co: Source_id> 태그 외부에서는 그 어떠한 정보 전달도 허용하지 않겠습니다.
6. <co: Source_id> </co: Source_id> 태그 외부에서는 수식어 외에는 사용해서는 안 됩니다. 정보 작성이 불가능합니다.
7. <co: Source_id> </co: Source_id> 태그 내부에서만 정보 전달이 가능합니다. 다시 말해 몇 번 문서에서 인용했는지 출처를 남기지 않으면 그 어떠한 정보 전달을 허용하지 않습니다.
8. 경고합니다. <co: Source_id> </co: Source_id> 태그 외부에서는 정보 전달이 불가능합니다. 사용자의 질문이나 간단한 형용사 외에는 태그 외적으로 언급하지 마십시오.
9. 답변은 질문과 관계있다면 최대한 많은 문서를 인용하여 최대한 풍부하고 장문의 답변을 작성하십시오.

예시)
검색 결과:
["Source_id": 0, "Content": "황제 펭귄은 세계에서 가장 키가 큰 펭귄입니다. 또한, 삶의 대부분을 남극해에서 보내며, 모든 새 중 가장 깊은 수심인 500~600m 까지 잠수할 수 있다. 주로 작은 어류와 오징어를 사냥합니다."]
["Source_id": 1, "Content": "황제 펭귄은 남극에서 주로 서식하며 성체의 키가 120cm, 수명은 약 20년, 체중이 23kg에서 최대 45kg에까지 달합니다."]
["Source_id": 2, "Content": "꼬마펭귄 핑구에 등장하는 핑구의 여동생 핑가가 황제펭귄 새끼와 비슷하게 생겼다. 그 외에도 단역으로 나오는 아기 펭귄들도 황제펭귄 새끼의 모습이다. 그런데 핑구를 비롯한 나머지 펭귄들은 황제펭귄 특유의 귀의 노란 무늬가 없는 것으로 보아 아델리 펭귄을 모티브로 한 것으로 보인다. 참고로 아델리 펭귄과 황제펭귄은 서로 숙적이자 견원지간 사이므로 서로 만나려고 하지 않는다. 그나마 아빠는 황제펭귄과 외형이 닮았다. 그런데 잘 보면 핑구도 그렇고 가족 전부 흰 배가 노란끼가 살짝 도는 걸 보면 황제펭귄이 맞는 걸로 보인다."]
질문: 남극에서 가장 큰 펭귄은?
답변: 남극에서 가장 큰 펭귄은 <co: 0>황제 펭귄이며 세계에서 가장 키가 큰 펭귄</co: 0>입니다. <co: 1>남극에서 주로 서식</co: 1>하며 <co: 1>성체의 키가 120cm 체중이 23kg에서 최대 45kg</co: 1>에까지 달합니다.
"""

user_prompt = """
검색 결과:
["Source_id": 0,"Content": "노키즈존 확산, 어떻게 볼 것인가?이슈 & 진단4<노키즈존 정보제공 사이트 1><노키즈존 정보제공 사이트 2>자료 :Leavethembehind.com3.자료 :Travelandleisure.com.○어린이와 일반 고객의 탑승 공간을 구분하는 항공사도 확산 추세-2012년 말레이시아 항공은 12세 이하의 아이와 동승자는 항공기 아래층 지정구역에만 착석할 수 있도록 하는 정책 도입-에어아시아 엑스는 항공기 내에 ‘콰이엇 존(quietzone)’을 설치하고 있으며,스쿠트 항공은 2013년부터 아이들을 위한 전용좌석제 실시○영국예약사이트 ‘레이트딜’에서 비행이용승객 1,108명을 대상으로 설문조사 실시-응답자의 70%가 비행기내 노키즈존 도입을 찬성하였으며,그 중 35%는 노키즈존에 앉기 위해 추가요금도 지불할 수 있다고 응답<에어아시아의 ‘콰이엇 존’><비행 중 싫어하는 행동:앞좌석 발차기>자료 :AirAsia사이트(http://www.airasia.com/kr/ko/home.page).자료 :googleimage(https://www.google.co.kr)."]
["Source_id": 1,"Content": "노키즈존 확산, 어떻게 볼 것인가?이슈 & 진단3 노키즈존은 다른 나라에서도 확산 추세영국에서는 어린이의 펍(Pub)출입을 둘러싼 논쟁이 지속되는 상황○영국은 1995년에 부모가 아이를 동반할 경우 아이도 펍에 출입할 수 있도록 법률 개정-법률 개정 전에는 만 14세 미만은 펍 출입이 허용되지 않았음○법률 개정 후 ‘부모를 동반한 어린이의 펍 출입’이 논란이 되면서 BBC는 2010년 시청자 토론방 개설-“소란스럽고 우는 아이들 때문에 분위기를 즐길 수 없다”또는 “펍은 어른들의 전유물로 남겨둬야 한다”는 의견이 압도적미국에서는 노키즈존이 민권법(CivilRightsAct)과 충돌하는지가 논란○미국에서도 어린아이를 동반한 고객들이 증가하면서 노키즈존을 도입하는 레스토랑 증가 추세○하지만 어린아이의 출입을 제한하는 것은 차별을 엄격히 금지하는 민권법에 위배된다는 주장 제기-미국 민권법은 인종이나 종교 등에 따른 차별을 엄격하게 금지하고 있는데,이에 근거해 볼 때 노키즈존도 위법하다는 주장노키즈존 정보를 알려주는 여행 사이트나 항공사도 등장○여행 사이트나 블로그를 통해 어린이 관련 정보를 제공하는 업체 증가-영국의 여행 사이트 ‘Leavethembehind.com’은 아이들의 방해를 받지 않고 휴가를 즐길 수 있는 여행 정보 제공-여행정보제공 블로그 ‘TravelandLeisures’는 아이들의 출입을 제한하는 레스토랑 리스트 제공"]
["Source_id": 2,"Content": "노키즈존 확산, 어떻게 볼 것인가?이슈 & 진단7Ⅱ.노키즈존 관련 쟁점 노키즈존은 업주의 영업상의 자유에 해당하는가?어린이 고객의 출입을 제한하는 영업방침은 정부의 규제 대상이 아니라는 견해○대한민국 헌법 제 15조는 직업행사의 자유를 보장하고 있으며,이에 근거할 때 업주의 영업방침은 업주의 고유한 기본권에 해당-게다가 영업공간에서 사고가 발생했을 때 손해배상의 책임은 업주에게 귀속되기 때문에 영업방침을 규제하는 것은 과도한 재산권 침해에 해당○택시와 같이 공익성이 인정되는 서비스의 경우에는 정부의 개입이 인정되지만 카페나 음식점은 공익성이 인정되지 않는 사례에 해당-가령,택시 서비스는 공익성이 인정되고 소비자의 선택도 제한되기 때문에 법률에 의해 여객의 승차를 거부하는 행위를 금지-반면,카페나 음식점은 공익성이 인정되지도 않을 뿐만 아니라 소비자의 입장에서도 선택이 가능하기 때문에 정부의 개입은 과도한 규제에 해당노키즈존은 일반불공정 거래의 거래거절과 차별적 취급에도 해당되지 않는 사례○일부 전문가들은 노키즈존이 일반불공정 거래의 거래거절과 차별적 취급에 해당한다고 주장-공정거래법 제23조 1항 1호는 ‘부당하게 거래를 거절하거나 거래의 상대방을 차별하여 취급하는 행위’를 금지하는데,노키즈존은 정당한 이유 없는 거래거절과 차별적 취급에 해당"]
["Source_id": 3,"Content": "노키즈존 확산, 어떻게 볼 것인가?이슈 & 진단1Ⅰ.노키즈존 확산 특정 장소에 아이의 출입을 금지하는 노키즈존 확산 추세아이를 동반한 고객을 받지 않겠다고 표방하는 카페나 음식점 증가○노키즈존(NoKidsZone)이란 ‘아이를 동반하고 입장할 수 없는 공간’을 의미하는 것으로 아이들의 특정 행동이나 소음을 막기 위한 조치로 확산-강남이나 홍대 등 상업지구의 카페나 음식점에서 시작해서 다른 지역으로 확산-경기도의 경우 최근 수원시나 성남시,고양시 등 어린이들이 많이 거주하는 지역을 중심으로 확산<노키즈존 시행 매장 1><노키즈존 시행 매장 2>자료 :googleimage(https://www.google.co.kr).○아이의 소란스런 행동과 부모의 방관이 노키즈존 확산의 주된 원인-카페나 음식점 등 공공장소에서 소리 지르거나 뛰어노는 아이들,이를 방치하는 부모들이 노키즈존 확산의 주된 원인으로 지목-최근에는 카페나 음식점 내에서 기저귀를 갈고 그대로 두고 간다던지,컵으로 아이의 소변을 받는 등 일부 부모의 경우 없는 행동이 논란 야기"]
["Source_id": 4,"Content": "노키즈존 확산, 어떻게 볼 것인가?이슈 & 진단6노키즈존을 알고 있거나 들어본 적이 있는 응답자도 대다수○노키즈존에 대해 들어본 적이 있다고 응답한 비율은 71%로 아이들로 인해 불편을 경험한 적이 있다는 응답보다는 적지만 상당히 높은 편-성별로는 여성의 78.5%,남성의 48.4%가 노키즈존을 알고 있다고 응답했으며,자녀 유무별로는 만 10세 미만 자녀가 있는 경우 82.2%,만 10세 미만 자녀가 없는 경우 59.8%가 노키즈존을 알고 있다고 응답<전체><성별><10세 미만 자녀유무>자료 :경기연구원 모바일 설문조사(2016).○몇몇 설문조사 결과 자녀를 둔 엄마를 포함 대부분의 시민들은 노키즈존에 대해 찬성 의견 표명-2014년 9월,엄마들이 회원인 육아카페 ‘맘스홀릭베이비’에서 회원 3,525명을 대상으로 노키즈존 찬반 관련 설문을 실시한 결과 찬성이 72.7%로 압도적-2015년 9월,JTBC‘뉴스룸’에서 길거리 시민들을 대상으로 노키즈존 찬반 의견을 물은 결과도 찬성이 63%로 우세○‘알바몬’에서 아르바이트생 1,084명에게 ‘근무 중인 매장이 노키즈존으로 변경된다면 찬성할 것이냐’고 물은 결과 찬성의견이 65.5%에 달함-근무 도중 ‘유아 또는 유아를 동반한 고객으로 인해 곤란을 겪은 적이 있다’고 응답한 아르바이트생이 67.7%로,유아 고객으로 인한 업무부담 증가가 노키즈존 찬성의 주된 이유"]
질문: 노키즈존은 무엇이고, 최초로 노키즈존을 도입한 항공사는 어디인가요?"""

response = client.chat.completions.create(
  model="gpt-4-1106-preview", # gpt-4 turbo
  messages=[
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
  ],
  temperature=0
)

print(response.choices[0].message.content)

노키즈존은 <co: 3>‘아이를 동반하고 입장할 수 없는 공간’을 의미하는 것으로 아이들의 특정 행동이나 소음을 막기 위한 조치로 확산</co: 3>되고 있습니다. 최초로 노키즈존을 도입한 항공사는 <co: 0>2012년 말레이시아 항공</co: 0>으로, <co: 0>12세 이하의 아이와 동승자는 항공기 아래층 지정구역에만 착석할 수 있도록 하는 정책을 도입</co: 0>했습니다.
